# Original token-chunk LM head — COMPONENT PREFLIGHT ONLY

This packet is an isolated frozen-head experiment, not a training submission or production/model installer. It tests B4/T133 and B4/T512, full V151936/H1536, frozen FP32 Parameter, native CUDA AMP FP16, and unchanged precision flags. Native reference uses the installed source-verified Transformers ForCausalLMLoss, TRL entropy, and Soup row_losses. Native TRL entropy already token-chunks128 rows; no nonexistent full entropy allocation is credited.

The original candidate chunks **tokens, never vocabulary**, recomputes each head tile in backward after the genuine upstream gradient arrives, casts dlogits to FP16, and performs one entire-vocabulary hidden adjoint per row. This avoids incorrect pre-scaled/precomputed FP16 gradients and partial vocabulary half-adjoint summation. Actual GradScaler.scale(65536) is exercised, including a full-vocabulary overflow-class witness. No optimizer or parameter step is run. The common detached-summary consumer is embedded byte-identically, and normalized FP32 loss identity is checked across actual Accelerate conversion. No duplicate consumer or trainer patch is installed.

37 local CPU tests pass on Torch2.15 CPU, not target CUDA2.11. Target numerical gates are conservative component engineering checks, not official task quality gates: metrics rtol1e-5/atol3e-6; hidden gradient relative L2<=0.001 and max absolute<=0.005*reference max+1e-7; exact first argmax, counts, NaN/Inf classes, input versions, and RNG state. Higher-order gradients are explicitly unsupported. Full versus token-sliced cuBLAS algorithm equivalence is tested, not assumed.

Numerical preflight, native measurement, and chunk measurement each run in a fresh protected child with an unchanged80s cap. The same synthetic inputs are generated in each method process. Bounded measurements report cold synchronized wall, three warm CUDA-event intervals, and allocated/reserved component peaks. Peaks include synthetic head weights/inputs but **are not full model/training memory**. Extra recomputation may make the candidate slower. No29s/3GB claim, task score, original Triton ownership credit, or admission inference. Full-task integration and paired quality/profile evidence are still required before any such claim.

Protected runtime/source/lock/pidfd/subreaper/quarantine/export helpers remain unchanged. Empty private cache paths; fixed native-then-chunk order. No task switch changes, model downloads, installs, credentials, cache prewarming, automatic retry, server submission or publication by this worker. Parent alone submits after independent review.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiU3RhbmRhbG9uZSBoZWFkIGNvbXBvbmVudCBvbmx5LiBObyBtb2RlbC9UcmFpbmVyIGluc3RhbGxhdGlvbiBvciBrZXJuZWwgY3JlZGl0LiIiIgo=', 'solution/token_chunk_head.py': 'IiIiT3JpZ2luYWwgbmF0aXZlLUdFTU0gdG9rZW4tY2h1bmsgY29tcG9uZW50OyBubyBtb2RlbCBwYXRjaCBvciBUcml0b24gY3JlZGl0LgoKVG9rZW4tYXhpcyBjaHVua3Mga2VlcCBlYWNoIGhpZGRlbi1ncmFkaWVudCByb3cncyBlbnRpcmUgdm9jYWJ1bGFyeSByZWR1Y3Rpb24uCkJhY2t3YXJkIHJlY29tcHV0ZXMgdGhlIGhlYWQgQUZURVIgdGhlIHJlYWwgdXBzdHJlYW0vR3JhZFNjYWxlciBzY2FsYXIgYXJyaXZlcy4KTm8gU291cCB2ZW5kb3IgY2h1bmsgaW1wbGVtZW50YXRpb24gaXMgaW1wb3J0ZWQgb3IgY29waWVkLiBOYXRpdmUgUHlUb3JjaCBvcHMKYXJlIHJlZmVyZW5jZSBwcmltaXRpdmVzIGFuZCBlYXJuIG5vIG9yaWdpbmFsLVRyaXRvbiBjcmVkaXQuIFRhcmdldCBzbGljZWQvZnVsbApHRU1NIGVxdWl2YWxlbmNlIGlzIHRlc3RlZCwgbm90IGFzc3VtZWQuIE5vIG1vZGVsL29wdGltaXplci90cmFpbmluZyBpcyBpbnN0YWxsZWQuCiIiIgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCkBkYXRhY2xhc3MKY2xhc3MgU3VtbWFyaWVzOgogICAgZW50cm9weTogdG9yY2guVGVuc29yCiAgICB0b2tlbl9ubGw6IHRvcmNoLlRlbnNvcgogICAgcHJlZGljdGlvbnM6IHRvcmNoLlRlbnNvcgoKCmRlZiBfY2hlY2soaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcik6CiAgICBhc3NlcnQgdHlwZShoKSBpcyB0b3JjaC5UZW5zb3IgYW5kIHR5cGUodykgaW4gKHRvcmNoLlRlbnNvcix0b3JjaC5ubi5QYXJhbWV0ZXIpIGFuZCB0eXBlKGxhYmVscykgaXMgdG9yY2guVGVuc29yCiAgICBhc3NlcnQgaC5kZXZpY2U9PXcuZGV2aWNlPT1sYWJlbHMuZGV2aWNlIGFuZCBoLmRldmljZS50eXBlIGluICgnY3B1JywnY3VkYScpCiAgICBpZiBoLmRldmljZS50eXBlPT0nY3VkYSc6CiAgICAgICAgYXNzZXJ0IGguc2hhcGVbLTFdPT0xNTM2IGFuZCB3LnNoYXBlPT0oMTUxOTM2LDE1MzYpLCAnVGFzay1zcGVjaWZpYyBDVURBIGdlb21ldHJ5JwogICAgICAgIGFzc2VydCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eShoLmRldmljZSk9PSg3LDUpLCAnVDQgY29tcG9uZW50IG9ubHknCiAgICAgICAgYXNzZXJ0IHRvcmNoLmlzX2F1dG9jYXN0X2VuYWJsZWQoJ2N1ZGEnKSBhbmQgdG9yY2guZ2V0X2F1dG9jYXN0X2R0eXBlKCdjdWRhJyk9PXRvcmNoLmZsb2F0MTYsICdOYXRpdmUgQU1QIEZQMTYgYm91bmRhcnkgb25seScKICAgIGFzc2VydCBoLmR0eXBlPT13LmR0eXBlPT10b3JjaC5mbG9hdDMyIGFuZCBsYWJlbHMuZHR5cGU9PXRvcmNoLmludDY0CiAgICBhc3NlcnQgaC5uZGltPT0zIGFuZCB3Lm5kaW09PTIgYW5kIGxhYmVscy5zaGFwZT09aC5zaGFwZVs6Ml0gYW5kIGguc2hhcGVbLTFdPT13LnNoYXBlWzFdCiAgICBhc3NlcnQgaC5pc19jb250aWd1b3VzKCkgYW5kIHcuaXNfY29udGlndW91cygpIGFuZCBsYWJlbHMuaXNfY29udGlndW91cygpCiAgICBhc3NlcnQgbm90IGFueSh4LmlzX25lZygpIG9yIHguaXNfY29uaigpIGZvciB4IGluIChoLHcsbGFiZWxzKSkKICAgIGFzc2VydCBub3Qgdy5yZXF1aXJlc19ncmFkLCdGcm96ZW4gaGVhZCBvbmx5OiBubyB2b2NhYnVsYXJ5L3dlaWdodC1ncmFkaWVudCBhY2N1bXVsYXRpb24gcHJvb2YnCiAgICBhc3NlcnQgaC51bnR5cGVkX3N0b3JhZ2UoKS5kYXRhX3B0cigpIT13LnVudHlwZWRfc3RvcmFnZSgpLmRhdGFfcHRyKCkKICAgIGFzc2VydCB0eXBlKGNodW5rKSBpcyBpbnQgYW5kIGNodW5rPjAgYW5kIGgubnVtZWwoKT4wIGFuZCB3LnNoYXBlWzBdPjAKICAgIGFzc2VydCBkZW5vbWluYXRvciBpcyBOb25lIG9yICh0eXBlKGRlbm9taW5hdG9yKSBpcyBpbnQgYW5kIGRlbm9taW5hdG9yPj0wKSBvciAodHlwZShkZW5vbWluYXRvcikgaXMgdG9yY2guVGVuc29yIGFuZCBkZW5vbWluYXRvci5kZXZpY2U9PWguZGV2aWNlIGFuZCBkZW5vbWluYXRvci5kdHlwZT09dG9yY2guaW50NjQgYW5kIGRlbm9taW5hdG9yLm5kaW09PTAgYW5kIG5vdCBkZW5vbWluYXRvci5yZXF1aXJlc19ncmFkKQogICAgIyBOYXRpdmUgcHJpbWl0aXZlcyByZXRhaW4gdGhlaXIgZXhjZXB0aW9uYWwtdmFsdWUgYmVoYXZpb3I7IG5vIGRhdGEtZGVwZW5kZW50CiAgICAjIHN1Y2Nlc3Mgcm91dGUsIHBhZGRpbmcgcmVtb3ZhbCwgZ2xvYmFsIEFNUC9wcmVjaXNpb24tc2V0dGluZyBjaGFuZ2UsIG9yIGNhY2hlLgoKCmRlZiBzaGlmdF90YXJnZXRzKGxhYmVscyk6CiAgICByZXR1cm4gRi5wYWQobGFiZWxzLCgwLDEpLHZhbHVlPS0xMDApWy4uLiwxOl0uY29udGlndW91cygpCgoKZGVmIG5hdGl2ZV9ib3VuZGFyeShoLHcsbGFiZWxzLGRlbm9taW5hdG9yPU5vbmUpOgogICAgIyBFeHBsaWNpdCBDUFUgZW11bGF0aW9uIG9mIENVREEgQU1QJ3MgRlAzMiAtPiBGUDE2IGxpbmVhciBib3VuZGFyeS4KICAgIHJhdz1GLmxpbmVhcihoLnRvKHRvcmNoLmZsb2F0MTYpLHcudG8odG9yY2guZmxvYXQxNikpCiAgICB0YXJnZXRzPXNoaWZ0X3RhcmdldHMobGFiZWxzKQogICAgbG9zcz1GLmNyb3NzX2VudHJvcHkocmF3LmZsb2F0KCkudmlldygtMSx3LnNoYXBlWzBdKSx0YXJnZXRzLnZpZXcoLTEpLGlnbm9yZV9pbmRleD0tMTAwLAogICAgICAgICAgICAgICAgICAgICAgICAgcmVkdWN0aW9uPSdtZWFuJyBpZiBkZW5vbWluYXRvciBpcyBOb25lIGVsc2UgJ3N1bScpCiAgICBpZiBkZW5vbWluYXRvciBpcyBub3QgTm9uZTpsb3NzPWxvc3MvZGVub21pbmF0b3IKICAgIHB1YmxpYz1yYXcuZmxvYXQoKQogICAgbG9ncD1GLmxvZ19zb2Z0bWF4KHB1YmxpYyxkaW09LTEpCiAgICBubGw9Ri5jcm9zc19lbnRyb3B5KHB1YmxpYy5yZXNoYXBlKC0xLHcuc2hhcGVbMF0pLHRhcmdldHMucmVzaGFwZSgtMSksaWdub3JlX2luZGV4PS0xMDAscmVkdWN0aW9uPSdub25lJykucmVzaGFwZV9hcyhsYWJlbHMpCiAgICByZXR1cm4gbG9zcyxTdW1tYXJpZXMoLShsb2dwLmV4cCgpKmxvZ3ApLnN1bSgtMSksbmxsLHB1YmxpYy5hcmdtYXgoLTEpKSxyYXcKCgpjbGFzcyBfVG9rZW5DaHVuayh0b3JjaC5hdXRvZ3JhZC5GdW5jdGlvbik6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZChjdHgsaCx3LGxhYmVscyxjaHVuayxkZW5vbWluYXRvcik6CiAgICAgICAgX2NoZWNrKGgsdyxsYWJlbHMsY2h1bmssZGVub21pbmF0b3IpCiAgICAgICAgYix0LGs9aC5zaGFwZTt2PXcuc2hhcGVbMF0KICAgICAgICAjIEZQMzIgaW5wdXRzIGltcGx5IHRoZXNlIGNhc3RzIGNyZWF0ZSBvd25lZCBzbmFwc2hvdHMsIG5vdCBhbGlhc2VzLgogICAgICAgIGgxNj1oLnRvKHRvcmNoLmZsb2F0MTYpLnJlc2hhcGUoLTEsaykKICAgICAgICB3MTY9dy50byh0b3JjaC5mbG9hdDE2KQogICAgICAgIHRhcmdldHM9c2hpZnRfdGFyZ2V0cyhsYWJlbHMpLnJlc2hhcGUoLTEpCiAgICAgICAgZW50cm9weT10b3JjaC5lbXB0eShiKnQsZHR5cGU9dG9yY2guZmxvYXQzMixkZXZpY2U9aC5kZXZpY2UpCiAgICAgICAgbmxsPXRvcmNoLmVtcHR5KGIqdCxkdHlwZT10b3JjaC5mbG9hdDMyLGRldmljZT1oLmRldmljZSkKICAgICAgICBwcmVkaWN0aW9ucz10b3JjaC5lbXB0eShiKnQsZHR5cGU9dG9yY2guaW50NjQsZGV2aWNlPWguZGV2aWNlKQogICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLGIqdCxjaHVuayk6CiAgICAgICAgICAgIHN0b3A9bWluKHN0YXJ0K2NodW5rLGIqdCkKICAgICAgICAgICAgbG9naXRzPUYubGluZWFyKGgxNltzdGFydDpzdG9wXSx3MTYpLmZsb2F0KCkKICAgICAgICAgICAgbG9ncD1GLmxvZ19zb2Z0bWF4KGxvZ2l0cyxkaW09LTEpCiAgICAgICAgICAgIGVudHJvcHlbc3RhcnQ6c3RvcF09LShsb2dwLmV4cCgpKmxvZ3ApLnN1bSgtMSkKICAgICAgICAgICAgbmxsW3N0YXJ0OnN0b3BdPUYuY3Jvc3NfZW50cm9weShsb2dpdHMsdGFyZ2V0c1tzdGFydDpzdG9wXSxpZ25vcmVfaW5kZXg9LTEwMCxyZWR1Y3Rpb249J25vbmUnKQogICAgICAgICAgICBwcmVkaWN0aW9uc1tzdGFydDpzdG9wXT1sb2dpdHMuYXJnbWF4KC0xKQogICAgICAgICAgICBkZWwgbG9naXRzLGxvZ3AKICAgICAgICBkaXZpc29yPSh0YXJnZXRzIT0tMTAwKS5zdW0oKSBpZiBkZW5vbWluYXRvciBpcyBOb25lIGVsc2UgZGVub21pbmF0b3IKICAgICAgICAjIFRoaXMgc21hbGwgZ2xvYmFsIHJlZHVjdGlvbiBjYW4gZGlmZmVyIGJ5IHJvdW5kaW5nIGZyb20gbmF0aXZlIE5MTC1zdW0uCiAgICAgICAgIyBJdCBpcyBOT1QgY2xhaW1lZCBiaXQtZXhhY3Q7IHRhc2sgdG9sZXJhbmNlcyBhbmQgdGFyZ2V0IGdhdGVzIHN0aWxsIGFwcGx5LgogICAgICAgIGxvc3M9bmxsLnN1bSgpL2Rpdmlzb3IKICAgICAgICAjIEEgY2FsbGVyLW93bmVkIGRlbm9taW5hdG9yIHRlbnNvciBoYXMgbmF0aXZlIHNhdmVkLXZlcnNpb24gc2VtYW50aWNzLgogICAgICAgICMgU2F2aW5nIGl0IChyYXRoZXIgdGhhbiBvbmx5IGFzc2lnbmluZyBjdHguZGl2aXNvcikgY2F0Y2hlcyBtdXRhdGlvbi4KICAgICAgICB0ZW5zb3JfZGl2aXNvcj1pc2luc3RhbmNlKGRpdmlzb3IsdG9yY2guVGVuc29yKQogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZChoMTYsdzE2LHRhcmdldHMsKigoZGl2aXNvciwpIGlmIHRlbnNvcl9kaXZpc29yIGVsc2UgKCkpKQogICAgICAgIGN0eC5zaGFwZT1oLnNoYXBlO2N0eC5jaHVuaz1jaHVuaztjdHguZGl2aXNvcj1Ob25lIGlmIHRlbnNvcl9kaXZpc29yIGVsc2UgZGl2aXNvcgogICAgICAgIGVudHJvcHk9ZW50cm9weS5yZXNoYXBlKGIsdCk7bmxsPW5sbC5yZXNoYXBlKGIsdCk7cHJlZGljdGlvbnM9cHJlZGljdGlvbnMucmVzaGFwZShiLHQpCiAgICAgICAgY3R4Lm1hcmtfbm9uX2RpZmZlcmVudGlhYmxlKGVudHJvcHksbmxsLHByZWRpY3Rpb25zKQogICAgICAgIHJldHVybiBsb3NzLGVudHJvcHksbmxsLHByZWRpY3Rpb25zCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGJhY2t3YXJkKGN0eCxncmFkX2xvc3MsX2VudHJvcHksX25sbCxfcHJlZGljdGlvbnMpOgogICAgICAgIGlmIHRvcmNoLmlzX2dyYWRfZW5hYmxlZCgpOnJhaXNlIFJ1bnRpbWVFcnJvcignSGlnaGVyLW9yZGVyIHJlcXVlc3RzIG5vdCBzdXBwb3J0ZWQgYnkgc3RhbmRhbG9uZSBjb21wb25lbnQnKQogICAgICAgIHNhdmVkPWN0eC5zYXZlZF90ZW5zb3JzCiAgICAgICAgaDE2LHcxNix0YXJnZXRzPXNhdmVkWzozXQogICAgICAgIGRpdmlzb3I9c2F2ZWRbM10gaWYgbGVuKHNhdmVkKT09NCBlbHNlIGN0eC5kaXZpc29yCiAgICAgICAgIyBNYWluIGxvc3MgZGl2aXNpb24ncyB1cHN0cmVhbSBncmFkaWVudCBoYXBwZW5zIEJFRk9SRSBuYXRpdmUgQ0UgLT4gaGFsZi4KICAgICAgICBub3JtYWxpemVkPWdyYWRfbG9zcy9kaXZpc29yCiAgICAgICAgcmVzdWx0PXRvcmNoLmVtcHR5X2xpa2UoaDE2LGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsaDE2LnNoYXBlWzBdLGN0eC5jaHVuayk6CiAgICAgICAgICAgIHN0b3A9bWluKHN0YXJ0K2N0eC5jaHVuayxoMTYuc2hhcGVbMF0pCiAgICAgICAgICAgIHdpdGggdG9yY2guZW5hYmxlX2dyYWQoKToKICAgICAgICAgICAgICAgIHJhdz1GLmxpbmVhcihoMTZbc3RhcnQ6c3RvcF0sdzE2KS5kZXRhY2goKS5yZXF1aXJlc19ncmFkXygpCiAgICAgICAgICAgICAgICBsb3NzX3N1bT1GLmNyb3NzX2VudHJvcHkocmF3LmZsb2F0KCksdGFyZ2V0c1tzdGFydDpzdG9wXSxpZ25vcmVfaW5kZXg9LTEwMCxyZWR1Y3Rpb249J3N1bScpCiAgICAgICAgICAgICAgICAoZHoxNiwpPXRvcmNoLmF1dG9ncmFkLmdyYWQobG9zc19zdW0scmF3LGdyYWRfb3V0cHV0cz1ub3JtYWxpemVkKQogICAgICAgICAgICAjIE9uZSBmdWxsLVYgZG90IHBlciBoaWRkZW4gcm93OyBvdXRwdXQgcm91bmRlZCB0byBGUDE2IEJFRk9SRSBmcDMyIGNhc3QuCiAgICAgICAgICAgIHJlc3VsdFtzdGFydDpzdG9wXT0oZHoxNiBAIHcxNikuZmxvYXQoKQogICAgICAgICAgICBkZWwgcmF3LGxvc3Nfc3VtLGR6MTYKICAgICAgICByZXR1cm4gcmVzdWx0LnJlc2hhcGUoY3R4LnNoYXBlKSxOb25lLE5vbmUsTm9uZSxOb25lCgoKZGVmIHRva2VuX2NodW5rX2JvdW5kYXJ5KGgsdyxsYWJlbHMsY2h1bms9MTI4LGRlbm9taW5hdG9yPU5vbmUsc3VtbWFyeV9zaW5rPU5vbmUpOgogICAgbG9zcyxlbnRyb3B5LG5sbCxwcmVkaWN0aW9ucz1fVG9rZW5DaHVuay5hcHBseShoLHcsbGFiZWxzLGNodW5rLGRlbm9taW5hdG9yKQogICAgaWYgc3VtbWFyeV9zaW5rIGlzIG5vdCBOb25lOgogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsodG9rZW5fbmxsPW5sbCx0b2tlbl9lbnRyb3B5PWVudHJvcHksZmlyc3RfYXJnbWF4PXByZWRpY3Rpb25zLHJvdXRlPSdzdHJlYW1pbmcnKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdTdHJlYW1pbmcgc3VtbWFyeSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgICAgIGlmIG5vdCBzdW1tYXJ5X3NpbmsuY29tcGxldGUobG9zcyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIG5vcm1hbGl6ZWQtbG9zcyBpZGVudGl0eSBwdWJsaWNhdGlvbiBmYWlsZWQnKQogICAgcmV0dXJuIGxvc3MsU3VtbWFyaWVzKGVudHJvcHksbmxsLHByZWRpY3Rpb25zKQoK', 'shared_consumer/__init__.py': '', 'shared_consumer/bridge_state.py': 'IiIiUGVyLWNhbGwgZGV0YWNoZWQgbWV0cmljIG93bmVyc2hpcC4gTm8gbW9kZWwvVHJhaW5lciBwYXRjaCBvciB0ZW5zb3IgdmFsdWVzIHJlYWQuCgpDYWxsZXIgbXVzdCBmaXJzdCB2ZXJpZnkgdGhlIG5hdGl2ZSBRd2VuLT5QRUZULT5BY2NlbGVyYXRlIHNvdXJjZS9jYWxsYWJsZSBjaGFpbi4KVGhhdCBwcm9vZiwgb25lIHByb2R1Y2VyLCBhbmQgb25lIGRpcmVjdCBIRi1wYXJlbnQgcmV0dXJuIGJpbmQgaGFsZiBzb3VyY2UgbG9naXRzCnRvIHRoZSBwdWJsaWMgRlAzMiB0ZW5zb3IuIFBFRlQgYnlwYXNzZXMgaW5uZXIgUXdlbiBtb2R1bGUgaG9va3MsIHNvIG5vbmUgYXJlIHVzZWQuCiIiIgpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyCmZyb20gY29udGV4dHZhcnMgaW1wb3J0IENvbnRleHRWYXIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCB3ZWFrcmVmCmltcG9ydCB0b3JjaAoKX0FDVElWRT1Db250ZXh0VmFyKCdzb3VwX2NlX21ldHJpY190cmFuc2FjdGlvbicsZGVmYXVsdD1Ob25lKQpfRklFTERTPSgndG9rZW5fZW50cm9weScsJ2ZpcnN0X2FyZ21heCcsJ3Rva2VuX25sbCcpCgoKZGVmIG1ldGFkYXRhKHQpOgogICAgcmV0dXJuICh0dXBsZSh0LnNoYXBlKSx0dXBsZSh0LnN0cmlkZSgpKSx0LmR0eXBlLHQuZGV2aWNlLHQuc3RvcmFnZV9vZmZzZXQoKSx0Ll92ZXJzaW9uKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIERldGFjaGVkU3VtbWFyeToKICAgIHRva2VuX25sbDogdG9yY2guVGVuc29yCiAgICB0b2tlbl9lbnRyb3B5OiB0b3JjaC5UZW5zb3IKICAgIGZpcnN0X2FyZ21heDogdG9yY2guVGVuc29yCiAgICByb3V0ZTogc3RyCgoKY2xhc3MgVHJhbnNhY3Rpb246CiAgICBkZWYgX19pbml0X18oc2VsZixvd25lcixsYWJlbHMsdm9jYWJ1bGFyeT1Ob25lKToKICAgICAgICBzZWxmLm93bmVyX3JlZj13ZWFrcmVmLnJlZihvd25lcikKICAgICAgICBzZWxmLmxhYmVsc19yZWY9d2Vha3JlZi5yZWYobGFiZWxzKQogICAgICAgIHNlbGYubGFiZWxzX21ldGFkYXRhPW1ldGFkYXRhKGxhYmVscykKICAgICAgICBzZWxmLnJhd19yZWY9Tm9uZTtzZWxmLnJhd19tZXRhZGF0YT1Ob25lO3NlbGYucHVibGljX3JlZj1Ob25lO3NlbGYucHVibGljX21ldGFkYXRhPU5vbmUKICAgICAgICBzZWxmLnNvdXJjZV9raW5kPU5vbmU7c2VsZi5zb3VyY2Vfc2hhcGU9Tm9uZTtzZWxmLnNvdXJjZV9kZXZpY2U9Tm9uZTtzZWxmLnZvY2FidWxhcnk9dm9jYWJ1bGFyeQogICAgICAgIHNlbGYubG9zc19yZWY9Tm9uZTtzZWxmLmxvc3NfbWV0YWRhdGE9Tm9uZTtzZWxmLmNvbXBsZXRlZD1GYWxzZQogICAgICAgIHNlbGYuc3VtbWFyeT1Ob25lO3NlbGYuc3VtbWFyeV9tZXRhZGF0YT17fTtzZWxmLnJlcXVlc3RlZD1GYWxzZTtzZWxmLm9mZmVyZWQ9RmFsc2UKICAgICAgICBzZWxmLmJvdW5kPUZhbHNlO3NlbGYuY2xvc2VkPUZhbHNlO3NlbGYuaW52YWxpZF9yZWFzb249Tm9uZTtzZWxmLmNvbnN1bWVkPXNldCgpCgogICAgZGVmIGludmFsaWRhdGUoc2VsZixyZWFzb24pOgogICAgICAgIHNlbGYuaW52YWxpZF9yZWFzb249cmVhc29uCiAgICAgICAgc2VsZi5zdW1tYXJ5PU5vbmU7c2VsZi5zdW1tYXJ5X21ldGFkYXRhLmNsZWFyKCkKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBvZmZlcihzZWxmLCosdG9rZW5fbmxsLHRva2VuX2VudHJvcHksZmlyc3RfYXJnbWF4LHJvdXRlKToKICAgICAgICBpZiBzZWxmLmNsb3NlZCBvciBfQUNUSVZFLmdldCgpIGlzIG5vdCBzZWxmIG9yIHNlbGYuaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYub2ZmZXJlZDoKICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdkdXBsaWNhdGVfcHJvZHVjZXJfb2ZmZXInKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLm9mZmVyZWQ9VHJ1ZQogICAgICAgIGxhYmVscz1zZWxmLmxhYmVsc19yZWYoKQogICAgICAgIGlmIHNlbGYuc291cmNlX3NoYXBlIGlzIE5vbmUgb3IgbGFiZWxzIGlzIE5vbmUgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGE6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgncHJvZHVjZXJfbGFiZWxfb3Jfc291cmNlX2NoYW5nZWQnKTtyZXR1cm4gRmFsc2UKICAgICAgICBleHBlY3RlZF9zaGFwZT1zZWxmLnNvdXJjZV9zaGFwZVs6LTFdO2V4cGVjdGVkX2RldmljZT1zZWxmLnNvdXJjZV9kZXZpY2UKICAgICAgICB2YWx1ZXM9eyd0b2tlbl9ubGwnOnRva2VuX25sbCwndG9rZW5fZW50cm9weSc6dG9rZW5fZW50cm9weSwnZmlyc3RfYXJnbWF4JzpmaXJzdF9hcmdtYXh9CiAgICAgICAgZm9yIG5hbWUsdmFsdWUgaW4gdmFsdWVzLml0ZW1zKCk6CiAgICAgICAgICAgIGR0eXBlPXRvcmNoLmludDY0IGlmIG5hbWU9PSdmaXJzdF9hcmdtYXgnIGVsc2UgdG9yY2guZmxvYXQzMgogICAgICAgICAgICBpZiAodHlwZSh2YWx1ZSkgaXMgbm90IHRvcmNoLlRlbnNvciBvciB2YWx1ZS5yZXF1aXJlc19ncmFkIG9yIHZhbHVlLmdyYWRfZm4gaXMgbm90IE5vbmUKICAgICAgICAgICAgICAgICAgICBvciB0dXBsZSh2YWx1ZS5zaGFwZSkhPWV4cGVjdGVkX3NoYXBlIG9yIHZhbHVlLmRldmljZSE9ZXhwZWN0ZWRfZGV2aWNlCiAgICAgICAgICAgICAgICAgICAgb3IgdmFsdWUuZHR5cGUhPWR0eXBlIG9yIG5vdCB2YWx1ZS5pc19jb250aWd1b3VzKCkgb3IgdmFsdWUuaXNfbmVnKCkgb3IgdmFsdWUuaXNfY29uaigpKToKICAgICAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnc3VtbWFyeV9tZXRhZGF0YTonK25hbWUpO3JldHVybiBGYWxzZQogICAgICAgIGlmIHJvdXRlIG5vdCBpbiAoJ2FsbF9zYWZlJywnbWl4ZWRfcm93cycsJ3N0cmVhbWluZycpOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ3N1bW1hcnlfcm91dGUnKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLnN1bW1hcnk9RGV0YWNoZWRTdW1tYXJ5KCoqdmFsdWVzLHJvdXRlPXJvdXRlKQogICAgICAgIHNlbGYuc3VtbWFyeV9tZXRhZGF0YT17bjptZXRhZGF0YSh2KSBmb3Igbix2IGluIHZhbHVlcy5pdGVtcygpfQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIGNvbXBsZXRlKHNlbGYsbG9zcyk6CiAgICAgICAgaWYgc2VsZi5jbG9zZWQgb3IgX0FDVElWRS5nZXQoKSBpcyBub3Qgc2VsZiBvciBzZWxmLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYuY29tcGxldGVkOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ2R1cGxpY2F0ZV9sb3NzX2NvbXBsZXRpb24nKTtyZXR1cm4gRmFsc2UKICAgICAgICBzZWxmLmNvbXBsZXRlZD1UcnVlCiAgICAgICAgaWYgKHNlbGYuc3VtbWFyeSBpcyBOb25lIG9yIHR5cGUobG9zcykgaXMgbm90IHRvcmNoLlRlbnNvciBvciBsb3NzLmR0eXBlIT10b3JjaC5mbG9hdDMyCiAgICAgICAgICAgICAgICBvciBsb3NzLm5kaW0hPTAgb3Igbm90IGxvc3MucmVxdWlyZXNfZ3JhZCBvciBsb3NzLmRldmljZSE9c2VsZi5zb3VyY2VfZGV2aWNlKToKICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdwcm9kdWNlcl9sb3NzX2NvbnRyYWN0Jyk7cmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5sb3NzX3JlZj13ZWFrcmVmLnJlZihsb3NzKQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIGJpbmRfb3V0cHV0cyhzZWxmLGxvc3Msb3V0cHV0cyk6CiAgICAgICAgaWYgc2VsZi5jbG9zZWQgb3IgX0FDVElWRS5nZXQoKSBpcyBub3Qgc2VsZiBvciBzZWxmLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBGYWxzZQogICAgICAgIGlmIHNlbGYuYm91bmQ6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnZHVwbGljYXRlX3B1YmxpY19iaW5kJyk7cmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5ib3VuZD1UcnVlCiAgICAgICAgaWYgKHNlbGYuc3VtbWFyeSBpcyBOb25lIG9yIHNlbGYubG9zc19yZWYgaXMgTm9uZSBvciBzZWxmLmxvc3NfcmVmKCkgaXMgbm90IGxvc3MKICAgICAgICAgICAgICAgIG9yIGdldGF0dHIob3V0cHV0cywnbG9zcycsTm9uZSkgaXMgbm90IGxvc3MpOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ25vX2NvbXBsZXRlZF9wcm9kdWNlcl9vcl9sb3NzX2lkZW50aXR5Jyk7cmV0dXJuIEZhbHNlCiAgICAgICAgbGFiZWxzPXNlbGYubGFiZWxzX3JlZigpO3B1YmxpY19sb2dpdHM9Z2V0YXR0cihvdXRwdXRzLCdsb2dpdHMnLE5vbmUpCiAgICAgICAgaWYgbGFiZWxzIGlzIE5vbmUgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGE6CiAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgncHVibGljX29yX2xhYmVsX21ldGFkYXRhJyk7cmV0dXJuIEZhbHNlCiAgICAgICAgaWYgc2VsZi5zb3VyY2Vfa2luZD09J21hdGVyaWFsaXplZCc6CiAgICAgICAgICAgIGlmICh0eXBlKHB1YmxpY19sb2dpdHMpIGlzIG5vdCB0b3JjaC5UZW5zb3Igb3IgcHVibGljX2xvZ2l0cy5kdHlwZSE9dG9yY2guZmxvYXQzMgogICAgICAgICAgICAgICAgICAgIG9yIHR1cGxlKHB1YmxpY19sb2dpdHMuc2hhcGUpIT1zZWxmLnNvdXJjZV9zaGFwZSBvciBwdWJsaWNfbG9naXRzLmRldmljZSE9c2VsZi5zb3VyY2VfZGV2aWNlCiAgICAgICAgICAgICAgICAgICAgb3Igbm90IHB1YmxpY19sb2dpdHMuaXNfY29udGlndW91cygpIG9yIHB1YmxpY19sb2dpdHMuaXNfbmVnKCkgb3IgcHVibGljX2xvZ2l0cy5pc19jb25qKCkpOgogICAgICAgICAgICAgICAgc2VsZi5pbnZhbGlkYXRlKCdwdWJsaWNfb3JfbGFiZWxfbWV0YWRhdGEnKTtyZXR1cm4gRmFsc2UKICAgICAgICAgICAgc2VsZi5wdWJsaWNfcmVmPXdlYWtyZWYucmVmKHB1YmxpY19sb2dpdHMpO3NlbGYucHVibGljX21ldGFkYXRhPW1ldGFkYXRhKHB1YmxpY19sb2dpdHMpCiAgICAgICAgZWxpZiBzZWxmLnNvdXJjZV9raW5kPT0nc3RyZWFtaW5nJzoKICAgICAgICAgICAgaWYgcHVibGljX2xvZ2l0cyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHNlbGYuaW52YWxpZGF0ZSgnc3RyZWFtaW5nX3VuZXhwZWN0ZWRfZnVsbF9sb2dpdHMnKTtyZXR1cm4gRmFsc2UKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmludmFsaWRhdGUoJ3Vua25vd25fc291cmNlX2tpbmQnKTtyZXR1cm4gRmFsc2UKICAgICAgICAjIE5hdGl2ZSBBY2NlbGVyYXRlIHByZXNlcnZlcyBGUDMyIHNjYWxhciBUZW5zb3IgaWRlbnRpdHkuIEhGJ3Mgb3B0aW9uYWwKICAgICAgICAjIG5hdGl2ZSBpbi1wbGFjZSBsb3NzIHNjYWxlIHJldGFpbnMgdGhhdCBpZGVudGl0eTsgc3RhbXAgaXRzIGN1cnJlbnQKICAgICAgICAjIG1ldGFkYXRhIG5vdywgbm90IGJlZm9yZSB0aGF0IGtub3duIG5hdGl2ZSBvcGVyYXRpb24uCiAgICAgICAgc2VsZi5sb3NzX21ldGFkYXRhPW1ldGFkYXRhKGxvc3MpCiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBkZWYgdGFrZShzZWxmLG5hbWUscHVibGljX2xvZ2l0cyxsYWJlbHMpOgogICAgICAgIGlmIChuYW1lIG5vdCBpbiBfRklFTERTIG9yIHNlbGYuY2xvc2VkIG9yIF9BQ1RJVkUuZ2V0KCkgaXMgbm90IHNlbGYKICAgICAgICAgICAgICAgIG9yIHNlbGYuaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmUgb3Igc2VsZi5zdW1tYXJ5IGlzIE5vbmUpOnJldHVybiBOb25lCiAgICAgICAgaWYgbmFtZSBpbiBzZWxmLmNvbnN1bWVkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5pbnZhbGlkYXRlKCdkdXBsaWNhdGVfY29uc3VtZXI6JytuYW1lKQogICAgICAgIG9yaWdpbmFsX2xhYmVscz1zZWxmLmxhYmVsc19yZWYoKQogICAgICAgIHB1YmxpY19vaz0oc2VsZi5zb3VyY2Vfa2luZD09InN0cmVhbWluZyIgYW5kIHB1YmxpY19sb2dpdHMgaXMgTm9uZSkgb3IgKAogICAgICAgICAgICBzZWxmLnNvdXJjZV9raW5kPT0ibWF0ZXJpYWxpemVkIiBhbmQgc2VsZi5wdWJsaWNfcmVmIGlzIG5vdCBOb25lCiAgICAgICAgICAgIGFuZCBzZWxmLnB1YmxpY19yZWYoKSBpcyBwdWJsaWNfbG9naXRzIGFuZCBtZXRhZGF0YShwdWJsaWNfbG9naXRzKT09c2VsZi5wdWJsaWNfbWV0YWRhdGEpCiAgICAgICAgbG9zcz1zZWxmLmxvc3NfcmVmKCkgaWYgc2VsZi5sb3NzX3JlZiBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgICAgICBpZiAobm90IHB1YmxpY19vayBvciBsb3NzIGlzIE5vbmUgb3IgbWV0YWRhdGEobG9zcykhPXNlbGYubG9zc19tZXRhZGF0YQogICAgICAgICAgICAgICAgb3Igb3JpZ2luYWxfbGFiZWxzIGlzIG5vdCBsYWJlbHMgb3IgbWV0YWRhdGEobGFiZWxzKSE9c2VsZi5sYWJlbHNfbWV0YWRhdGEpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5pbnZhbGlkYXRlKCdjb25zdW1lcl9pZGVudGl0eV9vcl92ZXJzaW9uJykKICAgICAgICBmb3IgZmllbGQgaW4gX0ZJRUxEUzoKICAgICAgICAgICAgaWYgbWV0YWRhdGEoZ2V0YXR0cihzZWxmLnN1bW1hcnksZmllbGQpKSE9c2VsZi5zdW1tYXJ5X21ldGFkYXRhW2ZpZWxkXToKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLmludmFsaWRhdGUoJ3N1bW1hcnlfbW9kaWZpZWQ6JytmaWVsZCkKICAgICAgICBzZWxmLmNvbnN1bWVkLmFkZChuYW1lKQogICAgICAgIHJldHVybiBnZXRhdHRyKHNlbGYuc3VtbWFyeSxuYW1lKQoKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICBzZWxmLmNsb3NlZD1UcnVlO3NlbGYuc3VtbWFyeT1Ob25lO3NlbGYuc3VtbWFyeV9tZXRhZGF0YS5jbGVhcigpCiAgICAgICAgc2VsZi5yYXdfcmVmPU5vbmU7c2VsZi5wdWJsaWNfcmVmPU5vbmU7c2VsZi5sb3NzX3JlZj1Ob25lCiAgICAgICAgc2VsZi5wcm9kdWNlcl9vd25lcl9yZWY9Tm9uZQogICAgICAgIGlmIGhhc2F0dHIoc2VsZiwnY2h1bmtlZF9wYWNrZXQnKTpkZWwgc2VsZi5jaHVua2VkX3BhY2tldAoKCkBjb250ZXh0bWFuYWdlcgpkZWYgc2NvcGUob3duZXIsbGFiZWxzLCosdm9jYWJ1bGFyeT1Ob25lKToKICAgIGlmIF9BQ1RJVkUuZ2V0KCkgaXMgbm90IE5vbmU6cmFpc2UgUnVudGltZUVycm9yKCdOZXN0ZWQgbWV0cmljIG93bmVyc2hpcCByZXF1aXJlcyBuYXRpdmUgc3VzcGVuZGVkIGZhbGxiYWNrJykKICAgIHRyYW5zYWN0aW9uPVRyYW5zYWN0aW9uKG93bmVyLGxhYmVscyx2b2NhYnVsYXJ5KTt0b2tlbj1fQUNUSVZFLnNldCh0cmFuc2FjdGlvbikKICAgIHRyeTp5aWVsZCB0cmFuc2FjdGlvbgogICAgZmluYWxseTp0cmFuc2FjdGlvbi5jbG9zZSgpO19BQ1RJVkUucmVzZXQodG9rZW4pCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiBzdXNwZW5kZWQoKToKICAgIHRva2VuPV9BQ1RJVkUuc2V0KE5vbmUpCiAgICB0cnk6eWllbGQKICAgIGZpbmFsbHk6X0FDVElWRS5yZXNldCh0b2tlbikKCgpkZWYgcmVxdWVzdF9zaW5rKG93bmVyLHJhd19sb2dpdHMsbGFiZWxzKToKICAgIHRyYW5zYWN0aW9uPV9BQ1RJVkUuZ2V0KCkKICAgIGlmIHRyYW5zYWN0aW9uIGlzIE5vbmUgb3IgdHJhbnNhY3Rpb24uY2xvc2VkIG9yIHRyYW5zYWN0aW9uLmludmFsaWRfcmVhc29uIGlzIG5vdCBOb25lOnJldHVybiBOb25lCiAgICBpZiB0cmFuc2FjdGlvbi5yZXF1ZXN0ZWQ6CiAgICAgICAgdHJhbnNhY3Rpb24uaW52YWxpZGF0ZSgnbXVsdGlwbGVfbG9zc19wcm9kdWNlcnMnKTtyZXR1cm4gTm9uZQogICAgaWYgKHRyYW5zYWN0aW9uLm93bmVyX3JlZigpIGlzIG5vdCBvd25lciBvciB0cmFuc2FjdGlvbi5sYWJlbHNfcmVmKCkgaXMgbm90IGxhYmVscwogICAgICAgICAgICBvciB0eXBlKHJhd19sb2dpdHMpIGlzIG5vdCB0b3JjaC5UZW5zb3Igb3IgcmF3X2xvZ2l0cy5kdHlwZSE9dG9yY2guZmxvYXQxNgogICAgICAgICAgICBvciByYXdfbG9naXRzLm5kaW0hPTMgb3IgcmF3X2xvZ2l0cy5zaGFwZVs6LTFdIT1sYWJlbHMuc2hhcGUKICAgICAgICAgICAgb3IgcmF3X2xvZ2l0cy5kZXZpY2UhPWxhYmVscy5kZXZpY2Ugb3Igbm90IHJhd19sb2dpdHMuaXNfY29udGlndW91cygpCiAgICAgICAgICAgIG9yIHJhd19sb2dpdHMuaXNfbmVnKCkgb3IgcmF3X2xvZ2l0cy5pc19jb25qKCkgb3IgbWV0YWRhdGEobGFiZWxzKSE9dHJhbnNhY3Rpb24ubGFiZWxzX21ldGFkYXRhKToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdwcm9kdWNlcl9pZGVudGl0eV9vcl9tZXRhZGF0YScpO3JldHVybiBOb25lCiAgICBpZiB0cmFuc2FjdGlvbi52b2NhYnVsYXJ5IGlzIG5vdCBOb25lIGFuZCByYXdfbG9naXRzLnNoYXBlWy0xXSE9dHJhbnNhY3Rpb24udm9jYWJ1bGFyeToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCJwcm9kdWNlcl92b2NhYnVsYXJ5Iik7cmV0dXJuIE5vbmUKICAgIHRyYW5zYWN0aW9uLnJlcXVlc3RlZD1UcnVlO3RyYW5zYWN0aW9uLnJhd19yZWY9d2Vha3JlZi5yZWYocmF3X2xvZ2l0cykKICAgIHRyYW5zYWN0aW9uLnJhd19tZXRhZGF0YT1tZXRhZGF0YShyYXdfbG9naXRzKTt0cmFuc2FjdGlvbi5zb3VyY2Vfa2luZD0ibWF0ZXJpYWxpemVkIgogICAgdHJhbnNhY3Rpb24uc291cmNlX3NoYXBlPXR1cGxlKHJhd19sb2dpdHMuc2hhcGUpO3RyYW5zYWN0aW9uLnNvdXJjZV9kZXZpY2U9cmF3X2xvZ2l0cy5kZXZpY2UKICAgIHJldHVybiBQcm9kdWNlclNpbmsodHJhbnNhY3Rpb24pCgoKZGVmIGN1cnJlbnQoKTpyZXR1cm4gX0FDVElWRS5nZXQoKQoKCmNsYXNzIFByb2R1Y2VyU2luazoKICAgIGRlZiBfX2luaXRfXyhzZWxmLHRyYW5zYWN0aW9uKTpzZWxmLnRyYW5zYWN0aW9uPXRyYW5zYWN0aW9uCiAgICBkZWYgX19jYWxsX18oc2VsZiwqKmZpZWxkcyk6cmV0dXJuIHNlbGYudHJhbnNhY3Rpb24ub2ZmZXIoKipmaWVsZHMpCiAgICBkZWYgY29tcGxldGUoc2VsZixsb3NzKTpyZXR1cm4gc2VsZi50cmFuc2FjdGlvbi5jb21wbGV0ZShsb3NzKQoKCmRlZiByZXF1ZXN0X3N0cmVhbWluZ19zaW5rKG93bmVyLGxhYmVscywqLHZvY2FidWxhcnkpOgogICAgdHJhbnNhY3Rpb249X0FDVElWRS5nZXQoKQogICAgaWYgdHJhbnNhY3Rpb24gaXMgTm9uZSBvciB0cmFuc2FjdGlvbi5jbG9zZWQgb3IgdHJhbnNhY3Rpb24uaW52YWxpZF9yZWFzb24gaXMgbm90IE5vbmU6cmV0dXJuIE5vbmUKICAgIGlmIHRyYW5zYWN0aW9uLnJlcXVlc3RlZDoKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdtdWx0aXBsZV9sb3NzX3Byb2R1Y2VycycpO3JldHVybiBOb25lCiAgICBpZiAodHJhbnNhY3Rpb24ub3duZXJfcmVmKCkgaXMgbm90IG93bmVyIG9yIHRyYW5zYWN0aW9uLmxhYmVsc19yZWYoKSBpcyBub3QgbGFiZWxzCiAgICAgICAgICAgIG9yIG1ldGFkYXRhKGxhYmVscykhPXRyYW5zYWN0aW9uLmxhYmVsc19tZXRhZGF0YSBvciB0eXBlKHZvY2FidWxhcnkpIGlzIG5vdCBpbnQKICAgICAgICAgICAgb3Igdm9jYWJ1bGFyeTw9MSBvciB0cmFuc2FjdGlvbi52b2NhYnVsYXJ5IT12b2NhYnVsYXJ5KToKICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkYXRlKCdzdHJlYW1pbmdfcHJvZHVjZXJfaWRlbnRpdHlfb3JfbWV0YWRhdGEnKTtyZXR1cm4gTm9uZQogICAgdHJhbnNhY3Rpb24ucmVxdWVzdGVkPVRydWU7dHJhbnNhY3Rpb24uc291cmNlX2tpbmQ9J3N0cmVhbWluZycKICAgIHRyYW5zYWN0aW9uLnNvdXJjZV9zaGFwZT10dXBsZShsYWJlbHMuc2hhcGUpKyh2b2NhYnVsYXJ5LCk7dHJhbnNhY3Rpb24uc291cmNlX2RldmljZT1sYWJlbHMuZGV2aWNlCiAgICByZXR1cm4gUHJvZHVjZXJTaW5rKHRyYW5zYWN0aW9uKQo=', 'shared_consumer/consumer.py': 'IiIiQm91bmRlZCBjb25zdW1lciBjb21wb25lbnQ7IG5vIHByb2R1Y3Rpb24gaW5zdGFsbGVyIG9yIGF1dG9tYXRpYyBwYXRjaGluZy4KClRoZSBjYWxsZXIgbXVzdCBlc3RhYmxpc2ggbmF0aXZlIHNvdXJjZSBvd25lcnNoaXAgYW5kIHN1cHBseSBhIG1ldGFkYXRhIGd1YXJkLgpPbmx5IGFuIG93bmVkIG91dGVyIFJld2luZCBsb3NzLW9ubHkgdHJhbnNhY3Rpb24gbWF5IHVzZSBkZXRhY2hlZCBzdW1tYXJpZXMuCk5hdGl2ZSBIRiBzdGlsbCBleGVjdXRlcyB0aGUgbW9kZWwgZXhhY3RseSBvbmNlLiBVbnN1cHBvcnRlZCBwcmUtZm9yd2FyZCBjYWxscwpyZW1haW4gZnVsbHkgbmF0aXZlOyBwb3N0LWZvcndhcmQgbWF0ZXJpYWxpemVkIG1pc3NlcyB1c2UgdGhlIFNBTUUgbG9naXRzLgoiIiIKZnJvbSBmdW5jdG9vbHMgaW1wb3J0IHdyYXBzCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQppbXBvcnQgd2Vha3JlZgppbXBvcnQgdG9yY2gKZnJvbSAuIGltcG9ydCBicmlkZ2Vfc3RhdGUgYXMgc3RhdGUKCgpkZWYgbWF0ZXJpYWxpemVkX3NpbmsoaW5uZXJfb3duZXIscmF3X2xvZ2l0cyxsYWJlbHMpOgogICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICBpZiB0cmFuc2FjdGlvbiBpcyBOb25lIG9yIGdldGF0dHIodHJhbnNhY3Rpb24sJ3Byb2R1Y2VyX293bmVyX3JlZicsbGFtYmRhOk5vbmUpKCkgaXMgbm90IGlubmVyX293bmVyOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXRlcj10cmFuc2FjdGlvbi5vd25lcl9yZWYoKQogICAgcmV0dXJuIHN0YXRlLnJlcXVlc3Rfc2luayhvdXRlcixyYXdfbG9naXRzLGxhYmVscykgaWYgb3V0ZXIgaXMgbm90IE5vbmUgZWxzZSBOb25lCgoKZGVmIHN0cmVhbWluZ19zaW5rKGlubmVyX293bmVyLGxhYmVscywqLHZvY2FidWxhcnkpOgogICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICBpZiB0cmFuc2FjdGlvbiBpcyBOb25lIG9yIGdldGF0dHIodHJhbnNhY3Rpb24sJ3Byb2R1Y2VyX293bmVyX3JlZicsbGFtYmRhOk5vbmUpKCkgaXMgbm90IGlubmVyX293bmVyOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXRlcj10cmFuc2FjdGlvbi5vd25lcl9yZWYoKQogICAgcmV0dXJuIHN0YXRlLnJlcXVlc3Rfc3RyZWFtaW5nX3Npbmsob3V0ZXIsbGFiZWxzLHZvY2FidWxhcnk9dm9jYWJ1bGFyeSkgaWYgb3V0ZXIgaXMgbm90IE5vbmUgZWxzZSBOb25lCgoKZGVmIG5hdGl2ZV9vcmRlcl9yb3dfbG9zc2VzKHRva2VuX25sbCxsYWJlbHMpOgogICAgIiIiTWF0Y2ggbmF0aXZlIHJvd19sb3NzZXMnIHBlci1leGFtcGxlIFQtMSBGUDMyIG1hc2svc3VtL2NsYW1wIG9yZGVyLiIiIgogICAgbG9zc2VzPVtdO2NvdW50cz1bXQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgdGFyZ2V0cz1sYWJlbHNbOiwxOl0KICAgICAgICBmb3IgaSBpbiByYW5nZSh0YXJnZXRzLnNpemUoMCkpOgogICAgICAgICAgICB0b2tlbl9jZT10b2tlbl9ubGxbaSw6LTFdLmZsb2F0KCkKICAgICAgICAgICAgbWFzaz0odGFyZ2V0c1tpXSE9LTEwMCkudG8odG9rZW5fY2UuZHR5cGUpCiAgICAgICAgICAgIHRva2Vucz1tYXNrLnN1bSgpO3RvdGFsPSh0b2tlbl9jZSptYXNrKS5zdW0oKQogICAgICAgICAgICBsb3NzZXMuYXBwZW5kKGZsb2F0KHRvdGFsL3Rva2Vucy5jbGFtcChtaW49MS4wKSkpCiAgICAgICAgICAgIGNvdW50cy5hcHBlbmQoaW50KHRva2Vucy5pdGVtKCkpKQogICAgcmV0dXJuIGxvc3Nlcyxjb3VudHMKCgpkZWYgcHVibGlzaF9tZXRyaWNzKHRyYWluZXIsb3V0cHV0cyxsYWJlbHMsYXR0ZW50aW9uX21hc2ssZW50cm9weV9mdW5jdGlvbix0cmFuc2FjdGlvbik6CiAgICBsb2dpdHM9b3V0cHV0cy5sb2dpdHMKICAgIGVudHJvcHk9dHJhbnNhY3Rpb24udGFrZSgndG9rZW5fZW50cm9weScsbG9naXRzLGxhYmVscykKICAgIHByZWRpY3Rpb25zPXRyYW5zYWN0aW9uLnRha2UoJ2ZpcnN0X2FyZ21heCcsbG9naXRzLGxhYmVscykKICAgIGlmIGVudHJvcHkgaXMgTm9uZSBvciBwcmVkaWN0aW9ucyBpcyBOb25lOgogICAgICAgIGlmIGxvZ2l0cyBpcyBOb25lOnJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIHN1bW1hcnkgYWJzZW50IG9yIGludmFsaWQgYmVmb3JlIG1ldHJpY3MnKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBlbnRyb3B5PWVudHJvcHlfZnVuY3Rpb24obG9naXRzKQogICAgICAgICAgICBwcmVkaWN0aW9ucz1sb2dpdHNbLi4uLCA6LTEsOl0uY29udGlndW91cygpLmFyZ21heChkaW09LTEpCiAgICBlbHNlOnByZWRpY3Rpb25zPXByZWRpY3Rpb25zWy4uLiw6LTFdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB2YWx1ZT0oZW50cm9weSphdHRlbnRpb25fbWFzaykuc3VtKCkvYXR0ZW50aW9uX21hc2suc3VtKCkKICAgICAgICB2YWx1ZT10cmFpbmVyLmFjY2VsZXJhdG9yLmdhdGhlcl9mb3JfbWV0cmljcyh2YWx1ZSkubWVhbigpLml0ZW0oKQogICAgdHJhaW5lci5fbWV0cmljc1sndHJhaW4nXVsnZW50cm9weSddLmFwcGVuZCh2YWx1ZSkKICAgIHRva2Vucz10cmFpbmVyLmFjY2VsZXJhdG9yLmdhdGhlcl9mb3JfbWV0cmljcyhhdHRlbnRpb25fbWFzay5zdW0oKSkuc3VtKCkuaXRlbSgpCiAgICB0cmFpbmVyLl90b3RhbF90cmFpbl90b2tlbnMrPXRva2VucwogICAgdHJhaW5lci5fbWV0cmljc1sndHJhaW4nXVsnbnVtX3Rva2VucyddPVt0cmFpbmVyLl90b3RhbF90cmFpbl90b2tlbnNdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB0YXJnZXRzPWxhYmVsc1suLi4sMTpdLmNvbnRpZ3VvdXMoKTttYXNrPXRhcmdldHMhPS0xMDAKICAgICAgICBjb3JyZWN0PSgocHJlZGljdGlvbnM9PXRhcmdldHMpJm1hc2spLnN1bSgpO3RvdGFsPW1hc2suc3VtKCkKICAgICAgICBjb3JyZWN0PXRyYWluZXIuYWNjZWxlcmF0b3IuZ2F0aGVyX2Zvcl9tZXRyaWNzKGNvcnJlY3QpCiAgICAgICAgdG90YWw9dHJhaW5lci5hY2NlbGVyYXRvci5nYXRoZXJfZm9yX21ldHJpY3ModG90YWwpCiAgICAgICAgdG90YWxfc3VtPXRvdGFsLnN1bSgpCiAgICAgICAgYWNjdXJhY3k9KGNvcnJlY3Quc3VtKCkvdG90YWxfc3VtKS5pdGVtKCkgaWYgdG90YWxfc3VtPjAgZWxzZSAwLjAKICAgIHRyYWluZXIuX21ldHJpY3NbJ3RyYWluJ11bJ21lYW5fdG9rZW5fYWNjdXJhY3knXS5hcHBlbmQoYWNjdXJhY3kpCgoKY2xhc3MgQ29uc3VtZXJJbnN0YWxsYXRpb246CiAgICAiIiJFeHBsaWNpdCBmaXh0dXJlL2ludGVncmF0aW9uIEFQSS4gTk9UIGEgc291cmNlLXZlcmlmeWluZyBpbnN0YWxsZXIuCgogICAgZ3VhcmQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHMpIHJldHVybnMgTm9uZSBvbmx5IHdoZW4gZXZlcnkKICAgIHByb2R1Y3Rpb24gb3duZXJzaGlwL2NvbmZpZ3VyYXRpb24gcmVxdWlyZW1lbnQgaGFzIGJlZW4gZXN0YWJsaXNoZWQuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCoscmV3aW5kX2NsYXNzLHNmdF9jbGFzcyxoZl9jbGFzcyxlbnRyb3B5X2Z1bmN0aW9uLAogICAgICAgICAgICAgICAgIHJld2luZF9yb3dzLHJld2luZF90b2tlbl9yb3dzLGd1YXJkLHZvY2FidWxhcnksY2h1bmtlZF9hdHRyLHByb2R1Y2VyX293bmVyPWxhbWJkYSBtb2RlbDptb2RlbCk6CiAgICAgICAgc2VsZi5yZXdpbmRfY2xhc3M9cmV3aW5kX2NsYXNzO3NlbGYuc2Z0X2NsYXNzPXNmdF9jbGFzcztzZWxmLmhmX2NsYXNzPWhmX2NsYXNzCiAgICAgICAgc2VsZi5vcmlnaW5hbF9yZXdpbmQ9cmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcwogICAgICAgIHNlbGYub3JpZ2luYWxfc2Z0PXNmdF9jbGFzcy5jb21wdXRlX2xvc3M7c2VsZi5vcmlnaW5hbF9oZj1oZl9jbGFzcy5jb21wdXRlX2xvc3MKICAgICAgICBzZWxmLm9yaWdpbmFsX3Jvd3M9cmV3aW5kX3Jvd3M7c2VsZi5vcmlnaW5hbF90b2tlbl9yb3dzPXJld2luZF90b2tlbl9yb3dzCiAgICAgICAgc2VsZi5nbG9iYWxzPXNlbGYub3JpZ2luYWxfcmV3aW5kLl9fZ2xvYmFsc19fO3NlbGYuY2h1bmtlZF9hdHRyPWNodW5rZWRfYXR0cgogICAgICAgIGlmIChzZWxmLmdsb2JhbHMuZ2V0KCdyb3dfbG9zc2VzJykgaXMgbm90IHJld2luZF9yb3dzCiAgICAgICAgICAgICAgICBvciBzZWxmLmdsb2JhbHMuZ2V0KCdyb3dfbG9zc2VzX2Zyb21fdG9rZW5fbG9zc2VzJykgaXMgbm90IHJld2luZF90b2tlbl9yb3dzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignTmF0aXZlIFJld2luZCBoZWxwZXIgb3duZXJzIHJlcXVpcmVkJykKICAgICAgICBvcmlnaW5hbF9yZXdpbmQ9c2VsZi5vcmlnaW5hbF9yZXdpbmQ7b3JpZ2luYWxfc2Z0PXNlbGYub3JpZ2luYWxfc2Z0CiAgICAgICAgb3JpZ2luYWxfaGY9c2VsZi5vcmlnaW5hbF9oZjtzZW50aW5lbD1vYmplY3QoKQoKICAgICAgICBAd3JhcHMob3JpZ2luYWxfcmV3aW5kKQogICAgICAgIGRlZiBvdXRlcih0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cz1GYWxzZSxudW1faXRlbXNfaW5fYmF0Y2g9Tm9uZSk6CiAgICAgICAgICAgIGludGFjdD0ocmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcyBpcyBvdXRlciBhbmQgc2Z0X2NsYXNzLmNvbXB1dGVfbG9zcyBpcyBzZnQKICAgICAgICAgICAgICAgIGFuZCBoZl9jbGFzcy5jb21wdXRlX2xvc3MgaXMgb3JpZ2luYWxfaGYgYW5kIHNlbGYuZ2xvYmFscy5nZXQoJ3Jvd19sb3NzZXMnKSBpcyByb3dzCiAgICAgICAgICAgICAgICBhbmQgc2VsZi5nbG9iYWxzLmdldCgncm93X2xvc3Nlc19mcm9tX3Rva2VuX2xvc3NlcycpIGlzIHRva2VuX3Jvd3MpCiAgICAgICAgICAgIHJlYXNvbj1ndWFyZCh0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cykgaWYgaW50YWN0IGVsc2UgJ2NvbnN1bWVyX293bmVyX2NoYW5nZWQnCiAgICAgICAgICAgIGlmIHJlYXNvbiBpcyBub3QgTm9uZSBvciByZXR1cm5fb3V0cHV0cyBvciBzdGF0ZS5jdXJyZW50KCkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB3aXRoIHN0YXRlLnN1c3BlbmRlZCgpOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnaW5hbF9yZXdpbmQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9cmV0dXJuX291dHB1dHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBwcmV2aW91cz1nZXRhdHRyKHRyYWluZXIsY2h1bmtlZF9hdHRyLHNlbnRpbmVsKTtvd25lZF9wYWNrZXQ9Tm9uZQogICAgICAgICAgICB3aXRoIHN0YXRlLnNjb3BlKG1vZGVsLGlucHV0c1snbGFiZWxzJ10sdm9jYWJ1bGFyeT12b2NhYnVsYXJ5KSBhcyB0cmFuc2FjdGlvbjoKICAgICAgICAgICAgICAgIHRyYW5zYWN0aW9uLnByb2R1Y2VyX293bmVyX3JlZj13ZWFrcmVmLnJlZihwcm9kdWNlcl9vd25lcihtb2RlbCkpCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0PW9yaWdpbmFsX3Jld2luZCh0cmFpbmVyLG1vZGVsLGlucHV0cyxyZXR1cm5fb3V0cHV0cz1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV9pdGVtc19pbl9iYXRjaD1udW1faXRlbXNfaW5fYmF0Y2gpCiAgICAgICAgICAgICAgICAgICAgaWYgdHJhbnNhY3Rpb24uc291cmNlX2tpbmQ9PSdzdHJlYW1pbmcnIGFuZCAoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0cmFuc2FjdGlvbi5pbnZhbGlkX3JlYXNvbiBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgb3IgdHJhbnNhY3Rpb24uY29uc3VtZWQhPXNldChzdGF0ZS5fRklFTERTKSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignU3RyZWFtaW5nIHN1bW1hcmllcyB3ZXJlIG5vdCBjb21wbGV0ZWx5IGNvbnN1bWVkJykKICAgICAgICAgICAgICAgICAgICByZXR1cm4gcmVzdWx0CiAgICAgICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgICAgIG93bmVkX3BhY2tldD1nZXRhdHRyKHRyYW5zYWN0aW9uLCdjaHVua2VkX3BhY2tldCcsTm9uZSkKICAgICAgICAgICAgICAgICAgICBpZiBvd25lZF9wYWNrZXQgaXMgbm90IE5vbmUgYW5kIGdldGF0dHIodHJhaW5lcixjaHVua2VkX2F0dHIsc2VudGluZWwpIGlzIG93bmVkX3BhY2tldDoKICAgICAgICAgICAgICAgICAgICAgICAgaWYgcHJldmlvdXMgaXMgc2VudGluZWw6ZGVsYXR0cih0cmFpbmVyLGNodW5rZWRfYXR0cikKICAgICAgICAgICAgICAgICAgICAgICAgZWxzZTpzZXRhdHRyKHRyYWluZXIsY2h1bmtlZF9hdHRyLHByZXZpb3VzKQoKICAgICAgICBAd3JhcHMob3JpZ2luYWxfc2Z0KQogICAgICAgIGRlZiBzZnQodHJhaW5lcixtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9RmFsc2UsbnVtX2l0ZW1zX2luX2JhdGNoPU5vbmUpOgogICAgICAgICAgICB0cmFuc2FjdGlvbj1zdGF0ZS5jdXJyZW50KCkKICAgICAgICAgICAgaWYgKHRyYW5zYWN0aW9uIGlzIE5vbmUgb3IgdHJhbnNhY3Rpb24ub3duZXJfcmVmKCkgaXMgbm90IG1vZGVsCiAgICAgICAgICAgICAgICAgICAgb3IgdHJhbnNhY3Rpb24ubGFiZWxzX3JlZigpIGlzIG5vdCBpbnB1dHMuZ2V0KCdsYWJlbHMnKSk6CiAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ2luYWxfc2Z0KHRyYWluZXIsbW9kZWwsaW5wdXRzLHJldHVybl9vdXRwdXRzPXJldHVybl9vdXRwdXRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBwYXJlbnQ9Z2V0YXR0cihzdXBlcihzZnRfY2xhc3MsdHJhaW5lciksJ2NvbXB1dGVfbG9zcycsTm9uZSkKICAgICAgICAgICAgaWYgZ2V0YXR0cihwYXJlbnQsJ19fZnVuY19fJyxOb25lKSBpcyBub3Qgb3JpZ2luYWxfaGY6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBIRiBwYXJlbnQgY2hhbmdlZCBpbnNpZGUgb3duZWQgdHJhbnNhY3Rpb24nKQogICAgICAgICAgICBsYWJlbHM9aW5wdXRzWydsYWJlbHMnXTttYXNrPWlucHV0c1snYXR0ZW50aW9uX21hc2snXQogICAgICAgICAgICBpbnB1dHMucG9wKCdfcHJlZGljdGlvbl9sb3NzX29ubHknLE5vbmUpO2lucHV0c1sndXNlX2NhY2hlJ109RmFsc2UKICAgICAgICAgICAgbG9zcyxvdXRwdXRzPXBhcmVudChtb2RlbCxpbnB1dHMscmV0dXJuX291dHB1dHM9VHJ1ZSxudW1faXRlbXNfaW5fYmF0Y2g9bnVtX2l0ZW1zX2luX2JhdGNoKQogICAgICAgICAgICBib3VuZD10cmFuc2FjdGlvbi5iaW5kX291dHB1dHMobG9zcyxvdXRwdXRzKQogICAgICAgICAgICBpZiBvdXRwdXRzLmxvZ2l0cyBpcyBOb25lOgogICAgICAgICAgICAgICAgaWYgbm90IGJvdW5kIG9yIHRyYW5zYWN0aW9uLnNvdXJjZV9raW5kIT0nc3RyZWFtaW5nJzoKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ01pc3NpbmcgZnVsbCBsb2dpdHMgd2l0aG91dCB2YWxpZGF0ZWQgc3RyZWFtaW5nIHN1bW1hcmllcycpCiAgICAgICAgICAgICAgICB0cmFuc2FjdGlvbi5jaHVua2VkX3BhY2tldD1TaW1wbGVOYW1lc3BhY2UodG9rZW5fbG9zcz10cmFuc2FjdGlvbi5zdW1tYXJ5LnRva2VuX25sbCkKICAgICAgICAgICAgICAgIHNldGF0dHIodHJhaW5lcixjaHVua2VkX2F0dHIsdHJhbnNhY3Rpb24uY2h1bmtlZF9wYWNrZXQpCiAgICAgICAgICAgIHB1Ymxpc2hfbWV0cmljcyh0cmFpbmVyLG91dHB1dHMsbGFiZWxzLG1hc2ssZW50cm9weV9mdW5jdGlvbix0cmFuc2FjdGlvbikKICAgICAgICAgICAgcmV0dXJuIChsb3NzLG91dHB1dHMpIGlmIHJldHVybl9vdXRwdXRzIGVsc2UgbG9zcwoKICAgICAgICBkZWYgcm93cyhsb2dpdHMsbGFiZWxzKToKICAgICAgICAgICAgdHJhbnNhY3Rpb249c3RhdGUuY3VycmVudCgpCiAgICAgICAgICAgIHRva2VuX25sbD10cmFuc2FjdGlvbi50YWtlKCd0b2tlbl9ubGwnLGxvZ2l0cyxsYWJlbHMpIGlmIHRyYW5zYWN0aW9uIGVsc2UgTm9uZQogICAgICAgICAgICByZXR1cm4gbmF0aXZlX29yZGVyX3Jvd19sb3NzZXModG9rZW5fbmxsLGxhYmVscykgaWYgdG9rZW5fbmxsIGlzIG5vdCBOb25lIGVsc2UgcmV3aW5kX3Jvd3MobG9naXRzLGxhYmVscykKCiAgICAgICAgZGVmIHRva2VuX3Jvd3ModG9rZW5fbG9zcyxsYWJlbHMpOgogICAgICAgICAgICB0cmFuc2FjdGlvbj1zdGF0ZS5jdXJyZW50KCkKICAgICAgICAgICAgaWYgdHJhbnNhY3Rpb24gaXMgTm9uZTpyZXR1cm4gcmV3aW5kX3Rva2VuX3Jvd3ModG9rZW5fbG9zcyxsYWJlbHMpCiAgICAgICAgICAgIGlmICh0cmFuc2FjdGlvbi5zb3VyY2Vfa2luZCE9J3N0cmVhbWluZycgb3IgdHJhbnNhY3Rpb24uc3VtbWFyeSBpcyBOb25lCiAgICAgICAgICAgICAgICAgICAgb3IgdG9rZW5fbG9zcyBpcyBub3QgdHJhbnNhY3Rpb24uc3VtbWFyeS50b2tlbl9ubGwpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdGb3JlaWduIHN0cmVhbWluZyB0b2tlbi1sb3NzIHBhY2tldCcpCiAgICAgICAgICAgIHRva2VuX25sbD10cmFuc2FjdGlvbi50YWtlKCd0b2tlbl9ubGwnLE5vbmUsbGFiZWxzKQogICAgICAgICAgICBpZiB0b2tlbl9ubGwgaXMgTm9uZTpyYWlzZSBSdW50aW1lRXJyb3IoJ0ludmFsaWQgc3RyZWFtaW5nIFJld2luZCBzdW1tYXJ5JykKICAgICAgICAgICAgcmV0dXJuIG5hdGl2ZV9vcmRlcl9yb3dfbG9zc2VzKHRva2VuX25sbCxsYWJlbHMpCgogICAgICAgIHNlbGYub3V0ZXI9b3V0ZXI7c2VsZi5zZnQ9c2Z0O3NlbGYucm93cz1yb3dzO3NlbGYudG9rZW5fcm93cz10b2tlbl9yb3dzCiAgICAgICAgcmV3aW5kX2NsYXNzLmNvbXB1dGVfbG9zcz1vdXRlcjtzZnRfY2xhc3MuY29tcHV0ZV9sb3NzPXNmdAogICAgICAgIHNlbGYuZ2xvYmFsc1sncm93X2xvc3NlcyddPXJvd3M7c2VsZi5nbG9iYWxzWydyb3dfbG9zc2VzX2Zyb21fdG9rZW5fbG9zc2VzJ109dG9rZW5fcm93cwoKICAgIGRlZiByZXN0b3JlKHNlbGYpOgogICAgICAgIHJlc3VsdD1UcnVlCiAgICAgICAgZm9yIG93bmVyLGtleSxvdXJzLG9sZCBpbiBbKHNlbGYucmV3aW5kX2NsYXNzLCdjb21wdXRlX2xvc3MnLHNlbGYub3V0ZXIsc2VsZi5vcmlnaW5hbF9yZXdpbmQpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChzZWxmLnNmdF9jbGFzcywnY29tcHV0ZV9sb3NzJyxzZWxmLnNmdCxzZWxmLm9yaWdpbmFsX3NmdCldOgogICAgICAgICAgICBpZiBnZXRhdHRyKG93bmVyLGtleSkgaXMgb3VyczpzZXRhdHRyKG93bmVyLGtleSxvbGQpCiAgICAgICAgICAgIGVsc2U6cmVzdWx0PUZhbHNlCiAgICAgICAgZm9yIGtleSxvdXJzLG9sZCBpbiBbKCdyb3dfbG9zc2VzJyxzZWxmLnJvd3Msc2VsZi5vcmlnaW5hbF9yb3dzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoJ3Jvd19sb3NzZXNfZnJvbV90b2tlbl9sb3NzZXMnLHNlbGYudG9rZW5fcm93cyxzZWxmLm9yaWdpbmFsX3Rva2VuX3Jvd3MpXToKICAgICAgICAgICAgaWYgc2VsZi5nbG9iYWxzLmdldChrZXkpIGlzIG91cnM6c2VsZi5nbG9iYWxzW2tleV09b2xkCiAgICAgICAgICAgIGVsc2U6cmVzdWx0PUZhbHNlCiAgICAgICAgcmV0dXJuIHJlc3VsdAo=', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': 'IiIiQm91bmRlZCBzdGFuZGFsb25lIGhlYWQgY29tcG9uZW50OiBubyBtb2RlbCBsb2FkLCBpbnN0YWxsZXIsIG9wdGltaXplciBvciB0cmFpbmluZy4KTmF0aXZlIHJlZmVyZW5jZXMgYXJlIHRoZSBwaW5uZWQgRm9yQ2F1c2FsTE1Mb3NzLCBUUkwgZW50cm9weSBhbmQgU291cCByb3dfbG9zc2VzLgpPbmx5IGlzb2xhdGVkIGhlYWQrcmVxdWlyZWQtbWV0cmljK2Fkam9pbnQgQ1VEQSB0aW1pbmdzIGFuZCBhbGxvY2F0b3IgcGVha3MgY291bnQuCiIiIgppbXBvcnQgYXJncGFyc2UsY29sbGVjdGlvbnMsZ2MsaGFzaGxpYixpbnNwZWN0LGpzb24sb3Msc3RhdGlzdGljcyx0aW1lLHRyYWNlYmFjayx3ZWFrcmVmCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgd3JpdGUocGF0aCx2YWx1ZSk6CiAgICB0bXA9cGF0aC53aXRoX25hbWUocGF0aC5uYW1lKycucGVuZGluZycpO3RtcC53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsaW5kZW50PTIsYWxsb3dfbmFuPUZhbHNlKSsnXG4nKTt0bXAucmVwbGFjZShwYXRoKQoKCmRlZiBtYWluKCk6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tbWV0aG9kJyxjaG9pY2VzPVsnbnVtZXJpY2FsJywnbmF0aXZlJywnY2h1bmsnXSxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2FyZ3M9cC5wYXJzZV9hcmdzKCkKICAgIG91dD1QYXRoKGFyZ3Mub3V0KTthc3NlcnQgbm90IG91dC5leGlzdHMoKQogICAgcmVwb3J0PWRpY3Qoc3RhdHVzPSdGQUlMRUQnLHNjb3BlPSdIRUFEX0NPTVBPTkVOVF9QUkVGTElHSFRfT05MWScsbWV0aG9kPWFyZ3MubWV0aG9kLHRyYWluaW5nX2V4ZWN1dGVkPUZhbHNlLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLG9yaWdpbmFsX3RyaXRvbl9jcmVkaXQ9RmFsc2Uscm93cz1bXSkKICAgIHdyaXRlKG91dCxyZXBvcnQpCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoLHRyaXRvbix0cmFuc2Zvcm1lcnMKICAgICAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMubG9zcy5sb3NzX3V0aWxzIGltcG9ydCBGb3JDYXVzYWxMTUxvc3MKICAgICAgICBmcm9tIHRybC50cmFpbmVyLnV0aWxzIGltcG9ydCBlbnRyb3B5X2Zyb21fbG9naXRzCiAgICAgICAgZnJvbSBzb3VwX2NsaS50cmFpbmVyLnJld2luZF9oZiBpbXBvcnQgcm93X2xvc3NlcwogICAgICAgIGZyb20gc29sdXRpb24udG9rZW5fY2h1bmtfaGVhZCBpbXBvcnQgdG9rZW5fY2h1bmtfYm91bmRhcnkKICAgICAgICBmcm9tIHNoYXJlZF9jb25zdW1lciBpbXBvcnQgYnJpZGdlX3N0YXRlCiAgICAgICAgZnJvbSBzaGFyZWRfY29uc3VtZXIuY29uc3VtZXIgaW1wb3J0IHN0cmVhbWluZ19zaW5rLHB1Ymxpc2hfbWV0cmljcyxuYXRpdmVfb3JkZXJfcm93X2xvc3NlcwogICAgICAgIGZyb20gYWNjZWxlcmF0ZS51dGlscy5vcGVyYXRpb25zIGltcG9ydCBjb252ZXJ0X3RvX2ZwMzIKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycy5tb2RlbGluZ19vdXRwdXRzIGltcG9ydCBDYXVzYWxMTU91dHB1dFdpdGhQYXN0CiAgICAgICAgYXNzZXJ0IHRvcmNoLl9fdmVyc2lvbl9fPT0nMi4xMS4wK2N1MTI4JyBhbmQgdHJpdG9uLl9fdmVyc2lvbl9fPT0nMy42LjAnIGFuZCB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX189PSc1LjE2LjEnCiAgICAgICAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnIGFuZCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKT09KDcsNSkKICAgICAgICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJykKICAgICAgICBleHBlY3RlZD1qc29uLmxvYWRzKChQYXRoKF9fZmlsZV9fKS5wYXJlbnQvJ25hdGl2ZV9zb3VyY2VfbWFuaWZlc3QuanNvbicpLnJlYWRfdGV4dCgpKQogICAgICAgIGZvciBmbiBpbiAoRm9yQ2F1c2FsTE1Mb3NzLGVudHJvcHlfZnJvbV9sb2dpdHMscm93X2xvc3Nlcyxjb252ZXJ0X3RvX2ZwMzIpOgogICAgICAgICAgICBwYXRoPVBhdGgoaW5zcGVjdC5nZXRmaWxlKGZuKSk7YXNzZXJ0IGhhc2hsaWIuc2hhMjU2KHBhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKT09ZXhwZWN0ZWRbcGF0aC5uYW1lXQogICAgICAgIGZsYWdzPWRpY3QobWF0bXVsX2FsbG93X3RmMzI9dG9yY2guYmFja2VuZHMuY3VkYS5tYXRtdWwuYWxsb3dfdGYzMixmcDE2X3JlZHVjZWRfcHJlY2lzaW9uX3JlZHVjdGlvbj10b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd19mcDE2X3JlZHVjZWRfcHJlY2lzaW9uX3JlZHVjdGlvbikKICAgICAgICByZXBvcnQudXBkYXRlKHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fLHRyaXRvbj10cml0b24uX192ZXJzaW9uX18sZ3B1PXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLHByZWNpc2lvbl9mbGFnc191bm1vZGlmaWVkPWZsYWdzLG5hdGl2ZV9zb3VyY2VzPWV4cGVjdGVkKQogICAgICAgIGc9dG9yY2guR2VuZXJhdG9yKGRldmljZT0nY3VkYScpLm1hbnVhbF9zZWVkKDgxMDkyMykKICAgICAgICB3PXRvcmNoLm5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbigxNTE5MzYsMTUzNixkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MzIsZ2VuZXJhdG9yPWcpKi4wMjUscmVxdWlyZXNfZ3JhZD1GYWxzZSkKICAgICAgICBvd25lcj10b3JjaC5ubi5Nb2R1bGUoKQogICAgICAgIGNsYXNzIEdhdGhlcjoKICAgICAgICAgICAgZGVmIGdhdGhlcl9mb3JfbWV0cmljcyhzZWxmLHZhbHVlKTpyZXR1cm4gdmFsdWUKICAgICAgICBhc3NlcnQgbm90IHcucmVxdWlyZXNfZ3JhZAogICAgICAgIGRlZiBmaXh0dXJlKGIsdCxpZ25vcmVkPUZhbHNlKToKICAgICAgICAgICAgaD0odG9yY2gucmFuZG4oYix0LDE1MzYsZGV2aWNlPSdjdWRhJyxnZW5lcmF0b3I9ZykqLjIpLnJlcXVpcmVzX2dyYWRfKCkKICAgICAgICAgICAgbGFiZWxzPXRvcmNoLnJhbmRpbnQoMCwxNTE5MzYsKGIsdCksZGV2aWNlPSdjdWRhJyxnZW5lcmF0b3I9ZyxkdHlwZT10b3JjaC5pbnQ2NCkKICAgICAgICAgICAgbGFiZWxzWzosMF09LTEwMDtsYWJlbHNbOiw6OjVdPS0xMDAKICAgICAgICAgICAgaWYgaWdub3JlZDpsYWJlbHMuZmlsbF8oLTEwMCkKICAgICAgICAgICAgbWFzaz10b3JjaC5vbmVzX2xpa2UobGFiZWxzKTttYXNrWzosLTFdPTAKICAgICAgICAgICAgcmV0dXJuIGgsbGFiZWxzLG1hc2sKICAgICAgICBkZWYgcnVuKG1ldGhvZCxoLGxhYmVscyxtYXNrLHNjYWxlLGRlbm9taW5hdG9yLGRldGFpbHM9RmFsc2UpOgogICAgICAgICAgICBpZiBtZXRob2Q9PSduYXRpdmUnOgogICAgICAgICAgICAgICAgIyBBY3R1YWwgbmF0aXZlIENVREEgYXV0b2Nhc3QsIG5vdCBtZXJlbHkgYW4gZXhwbGljaXQtY2FzdCByZWZlcmVuY2UuCiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KCdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KToKICAgICAgICAgICAgICAgICAgICByYXc9Ri5saW5lYXIoaCx3KQogICAgICAgICAgICAgICAgICAgIGxvc3M9Rm9yQ2F1c2FsTE1Mb3NzKHJhdyxsYWJlbHMsdm9jYWJfc2l6ZT0xNTE5MzYsbnVtX2l0ZW1zX2luX2JhdGNoPWRlbm9taW5hdG9yKQogICAgICAgICAgICAgICAgYXNzZXJ0IHJhdy5kdHlwZT09dG9yY2guZmxvYXQxNiBhbmQgbG9zcy5kdHlwZT09dG9yY2guZmxvYXQzMgogICAgICAgICAgICAgICAgcHVibGljPXJhdy5mbG9hdCgpCiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgICAgICBlbnQ9ZW50cm9weV9mcm9tX2xvZ2l0cyhwdWJsaWMpCiAgICAgICAgICAgICAgICAgICAgZW50cm9weT0oKGVudCptYXNrKS5zdW0oKS9tYXNrLnN1bSgpKS5pdGVtKCkKICAgICAgICAgICAgICAgICAgICBzaGlmdD1wdWJsaWNbOiw6LTEsOl0uY29udGlndW91cygpCiAgICAgICAgICAgICAgICAgICAgcHJlZGljdGlvbnM9c2hpZnQuYXJnbWF4KC0xKTt0YXJnZXQ9bGFiZWxzWzosMTpdO3ZhbGlkPXRhcmdldCE9LTEwMAogICAgICAgICAgICAgICAgICAgIGNvcnJlY3Q9KChwcmVkaWN0aW9ucz09dGFyZ2V0KSZ2YWxpZCkuc3VtKCk7dG90YWw9dmFsaWQuc3VtKCkKICAgICAgICAgICAgICAgICAgICBhY2N1cmFjeT0oY29ycmVjdC90b3RhbCkuaXRlbSgpIGlmIHRvdGFsPjAgZWxzZSAwLgogICAgICAgICAgICAgICAgICAgIGRlbCBzaGlmdAogICAgICAgICAgICAgICAgICAgIHJvd3MsY291bnRzPXJvd19sb3NzZXMocHVibGljLGxhYmVscykKICAgICAgICAgICAgICAgICAgICBubGw9Tm9uZQogICAgICAgICAgICAgICAgICAgIGlmIGRldGFpbHM6CiAgICAgICAgICAgICAgICAgICAgICAgIG5sbD10b3JjaC5zdGFjayhbRi5jcm9zc19lbnRyb3B5KHB1YmxpY1tpLDotMV0sbGFiZWxzW2ksMTpdLGlnbm9yZV9pbmRleD0tMTAwLHJlZHVjdGlvbj0nbm9uZScpIGZvciBpIGluIHJhbmdlKGxhYmVscy5zaGFwZVswXSldKQogICAgICAgICAgICAgICAgZGVsIHB1YmxpYyxyYXcKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHRyYWluZXI9U2ltcGxlTmFtZXNwYWNlKF9tZXRyaWNzPXsndHJhaW4nOmNvbGxlY3Rpb25zLmRlZmF1bHRkaWN0KGxpc3QpfSxfdG90YWxfdHJhaW5fdG9rZW5zPTAsYWNjZWxlcmF0b3I9R2F0aGVyKCkpCiAgICAgICAgICAgICAgICB3aXRoIGJyaWRnZV9zdGF0ZS5zY29wZShvd25lcixsYWJlbHMsdm9jYWJ1bGFyeT0xNTE5MzYpIGFzIHRyYW5zYWN0aW9uOgogICAgICAgICAgICAgICAgICAgIHRyYW5zYWN0aW9uLnByb2R1Y2VyX293bmVyX3JlZj13ZWFrcmVmLnJlZihvd25lcikKICAgICAgICAgICAgICAgICAgICBzaW5rPXN0cmVhbWluZ19zaW5rKG93bmVyLGxhYmVscyx2b2NhYnVsYXJ5PTE1MTkzNikKICAgICAgICAgICAgICAgICAgICBhc3NlcnQgc2luayBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoJ2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICAgICAgICAgICAgICBsb3NzLHM9dG9rZW5fY2h1bmtfYm91bmRhcnkoaCx3LGxhYmVscyxjaHVuaz0xMjgsZGVub21pbmF0b3I9ZGVub21pbmF0b3Isc3VtbWFyeV9zaW5rPXNpbmspCiAgICAgICAgICAgICAgICAgICAgb3V0cHV0cz1jb252ZXJ0X3RvX2ZwMzIoQ2F1c2FsTE1PdXRwdXRXaXRoUGFzdChsb3NzPWxvc3MsbG9naXRzPU5vbmUpKQogICAgICAgICAgICAgICAgICAgIGFzc2VydCB0cmFuc2FjdGlvbi5iaW5kX291dHB1dHMobG9zcyxvdXRwdXRzKSBhbmQgb3V0cHV0cy5sb3NzIGlzIGxvc3MKICAgICAgICAgICAgICAgICAgICBwdWJsaXNoX21ldHJpY3ModHJhaW5lcixvdXRwdXRzLGxhYmVscyxtYXNrLGVudHJvcHlfZnJvbV9sb2dpdHMsdHJhbnNhY3Rpb24pCiAgICAgICAgICAgICAgICAgICAgc2F2ZWRfbmxsPXRyYW5zYWN0aW9uLnRha2UoJ3Rva2VuX25sbCcsTm9uZSxsYWJlbHMpCiAgICAgICAgICAgICAgICAgICAgYXNzZXJ0IHNhdmVkX25sbCBpcyBzLnRva2VuX25sbAogICAgICAgICAgICAgICAgICAgIHJvd3MsY291bnRzPW5hdGl2ZV9vcmRlcl9yb3dfbG9zc2VzKHNhdmVkX25sbCxsYWJlbHMpCiAgICAgICAgICAgICAgICAgICAgYXNzZXJ0IHRyYW5zYWN0aW9uLmNvbnN1bWVkPT1zZXQoYnJpZGdlX3N0YXRlLl9GSUVMRFMpCiAgICAgICAgICAgICAgICBlbnRyb3B5PXRyYWluZXIuX21ldHJpY3NbJ3RyYWluJ11bJ2VudHJvcHknXVswXQogICAgICAgICAgICAgICAgYWNjdXJhY3k9dHJhaW5lci5fbWV0cmljc1sndHJhaW4nXVsnbWVhbl90b2tlbl9hY2N1cmFjeSddWzBdCiAgICAgICAgICAgICAgICBlbnQ9cy5lbnRyb3B5O3ByZWRpY3Rpb25zPXMucHJlZGljdGlvbnNbOiw6LTFdO25sbD1zLnRva2VuX25sbFs6LDotMV0gaWYgZGV0YWlscyBlbHNlIE5vbmUKICAgICAgICAgICAgIyBBY3R1YWwgR3JhZFNjYWxlci5zY2FsZSBzdXBwbGllcyA2NTUzNiwgd2l0aG91dCBhbiBvcHRpbWl6ZXIvc3RlcC4KICAgICAgICAgICAgIyBPdGhlciBjYXNlcyByZXRhaW4gdGhlaXIgZ2VudWluZSBhcmJpdHJhcnkgdXBzdHJlYW0gc2NhbGFyLgogICAgICAgICAgICBpZiBzY2FsZT09NjU1MzYuOgogICAgICAgICAgICAgICAgc2NhbGVyPXRvcmNoLmFtcC5HcmFkU2NhbGVyKCdjdWRhJyxpbml0X3NjYWxlPTY1NTM2LikKICAgICAgICAgICAgICAgIHNjYWxlZD1zY2FsZXIuc2NhbGUobG9zcykKICAgICAgICAgICAgICAgIChkaCwpPXRvcmNoLmF1dG9ncmFkLmdyYWQoc2NhbGVkLGgpCiAgICAgICAgICAgICAgICBhc3NlcnQgc2NhbGVyLmdldF9zY2FsZSgpPT02NTUzNi4KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIChkaCwpPXRvcmNoLmF1dG9ncmFkLmdyYWQobG9zcyxoLHRvcmNoLnRlbnNvcihzY2FsZSxkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MzIpKQogICAgICAgICAgICByZXR1cm4gZGljdChsb3NzPWxvc3MuZGV0YWNoKCksZGg9ZGgsZW50cm9weT1lbnRyb3B5LGFjY3VyYWN5PWFjY3VyYWN5LHJvd3M9cm93cyxjb3VudHM9Y291bnRzLHByZWRpY3Rpb25zPXByZWRpY3Rpb25zLHRva2VuX2VudHJvcHk9ZW50LHRva2VuX25sbD1ubGwpCiAgICAgICAgZGVmIGNwdSh2YWx1ZSk6CiAgICAgICAgICAgIHJldHVybiB7azoodi5kZXRhY2goKS5jcHUoKSBpZiBpc2luc3RhbmNlKHYsdG9yY2guVGVuc29yKSBlbHNlIHYpIGZvciBrLHYgaW4gdmFsdWUuaXRlbXMoKX0KICAgICAgICBkZWYgY29tcGFyZShhLGIsbmFtZSk6CiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGEsdG9yY2guVGVuc29yKTphPXRvcmNoLnRlbnNvcihhKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShiLHRvcmNoLlRlbnNvcik6Yj10b3JjaC50ZW5zb3IoYikKICAgICAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKHRvcmNoLmlzbmFuKGEpLHRvcmNoLmlzbmFuKGIpKSxuYW1lKycgTmFOIGNsYXNzJwogICAgICAgICAgICBhc3NlcnQgdG9yY2guZXF1YWwodG9yY2guaXNwb3NpbmYoYSksdG9yY2guaXNwb3NpbmYoYikpIGFuZCB0b3JjaC5lcXVhbCh0b3JjaC5pc25lZ2luZihhKSx0b3JjaC5pc25lZ2luZihiKSksbmFtZSsnIGluZmluaXR5IGNsYXNzJwogICAgICAgICAgICBmaW5pdGU9dG9yY2guaXNmaW5pdGUoYSkmdG9yY2guaXNmaW5pdGUoYikKICAgICAgICAgICAgYWY9YVtmaW5pdGVdLmZsb2F0KCk7YmY9YltmaW5pdGVdLmZsb2F0KCk7ZGVsdGE9KGFmLWJmKS5hYnMoKQogICAgICAgICAgICBtYXhhYnM9ZGVsdGEubWF4KCkuaXRlbSgpIGlmIGRlbHRhLm51bWVsKCkgZWxzZSAwLgogICAgICAgICAgICByZWZtYXg9YmYuYWJzKCkubWF4KCkuaXRlbSgpIGlmIGJmLm51bWVsKCkgZWxzZSAwLgogICAgICAgICAgICByZWxfbDI9KGRlbHRhLm5vcm0oKS9iZi5ub3JtKCkuY2xhbXAobWluPTFlLTMwKSkuaXRlbSgpIGlmIGRlbHRhLm51bWVsKCkgZWxzZSAwLgogICAgICAgICAgICBpZiBuYW1lPT0nZGgnOgogICAgICAgICAgICAgICAgYXNzZXJ0IHJlbF9sMjw9LjAwMSBhbmQgbWF4YWJzPD0uMDA1KnJlZm1heCsxZS03LChuYW1lLHJlbF9sMixtYXhhYnMscmVmbWF4KQogICAgICAgICAgICBlbHNlOnRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGEsYixydG9sPTFlLTUsYXRvbD0zZS02LGVxdWFsX25hbj1UcnVlKQogICAgICAgICAgICByZXR1cm4gZGljdChtYXhfYWJzPW1heGFicyxyZWxhdGl2ZV9sMj1yZWxfbDIscmVmZXJlbmNlX2Fic19tYXg9cmVmbWF4LGV4YWN0X3ZhbHVlcz1ib29sKHRvcmNoLmVxdWFsKGEsYikpLHplcm9fc2lnbl9kaXNhZ3JlZW1lbnRzPWludCgoKGE9PTApJihiPT0wKSYodG9yY2guc2lnbmJpdChhKSE9dG9yY2guc2lnbmJpdChiKSkpLnN1bSgpKSkKICAgICAgICBpZiBhcmdzLm1ldGhvZD09J251bWVyaWNhbCc6CiAgICAgICAgICAgIGNhc2VzPVsoNCwxMzMsMS4sRmFsc2UsVHJ1ZSxGYWxzZSksKDQsNTEyLDY1NTM2LixGYWxzZSxUcnVlLEZhbHNlKSwoMSwzLC4xMjUsRmFsc2UsRmFsc2UsRmFsc2UpLCgxLDMsNjU1MzYuLFRydWUsRmFsc2UsRmFsc2UpLCgxLDIsNjU1MzYuLEZhbHNlLEZhbHNlLFRydWUpXQogICAgICAgICAgICBmb3IgYix0LHNjYWxlLGlnbm9yZWQsZXh0ZXJuYWxfZGVub20sb3ZlcmZsb3cgaW4gY2FzZXM6CiAgICAgICAgICAgICAgICBoLGxhYmVscyxtYXNrPWZpeHR1cmUoYix0LGlnbm9yZWQpO2Rlbm9tPXRvcmNoLnRlbnNvcihiKnQsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5pbnQ2NCkgaWYgZXh0ZXJuYWxfZGVub20gZWxzZSBOb25lCiAgICAgICAgICAgICAgICBpZiBvdmVyZmxvdzoKICAgICAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgICAgICAgICAgdy56ZXJvXygpO3dbMCwwXT0xLjt3WzEsMF09LTEuO2guemVyb18oKTtsYWJlbHMuZmlsbF8oLTEwMCk7bGFiZWxzWzAsMV09MAogICAgICAgICAgICAgICAgcm5nPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCk7dmVyc2lvbnM9KGguX3ZlcnNpb24sdy5fdmVyc2lvbixsYWJlbHMuX3ZlcnNpb24sbWFzay5fdmVyc2lvbikKICAgICAgICAgICAgICAgIG5hdGl2ZT1jcHUocnVuKCduYXRpdmUnLGgsbGFiZWxzLG1hc2ssc2NhbGUsZGVub20sVHJ1ZSkpO2NhbmRpZGF0ZT1jcHUocnVuKCdjaHVuaycsaCxsYWJlbHMsbWFzayxzY2FsZSxkZW5vbSxUcnVlKSkKICAgICAgICAgICAgICAgIGlmIG92ZXJmbG93OmFzc2VydCBub3QgdG9yY2guaXNmaW5pdGUobmF0aXZlWydkaCddKS5hbGwoKSwnQWN0dWFsIHNjYWxlZCBuYXRpdmUgb3ZlcmZsb3cgd2l0bmVzcyBkaWQgbm90IHRyaWdnZXInCiAgICAgICAgICAgICAgICBjaGVja3M9e2tleTpjb21wYXJlKGNhbmRpZGF0ZVtrZXldLG5hdGl2ZVtrZXldLGtleSkgZm9yIGtleSBpbiAoJ2xvc3MnLCdkaCcsJ2VudHJvcHknLCdhY2N1cmFjeScsJ3Jvd3MnLCd0b2tlbl9lbnRyb3B5JywndG9rZW5fbmxsJyl9CiAgICAgICAgICAgICAgICBhc3NlcnQgY2FuZGlkYXRlWydjb3VudHMnXT09bmF0aXZlWydjb3VudHMnXQogICAgICAgICAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGNhbmRpZGF0ZVsncHJlZGljdGlvbnMnXSxuYXRpdmVbJ3ByZWRpY3Rpb25zJ10pLCdhcmdtYXggZGlmZmVycycKICAgICAgICAgICAgICAgIGFzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpIGFuZCB2ZXJzaW9ucz09KGguX3ZlcnNpb24sdy5fdmVyc2lvbixsYWJlbHMuX3ZlcnNpb24sbWFzay5fdmVyc2lvbikKICAgICAgICAgICAgICAgIHJlcG9ydFsncm93cyddLmFwcGVuZChkaWN0KEI9YixUPXQsVj0xNTE5MzYsSD0xNTM2LHNjYWxlPXNjYWxlLGFsbF9pZ25vcmVkPWlnbm9yZWQsZXh0ZXJuYWxfZGVub21pbmF0b3I9ZXh0ZXJuYWxfZGVub20sc2NhbGVkX292ZXJmbG93X3dpdG5lc3M9b3ZlcmZsb3csYWN0dWFsX2N1ZGFfZ3JhZHNjYWxlcj1zY2FsZT09NjU1MzYuLGNoZWNrcz1jaGVja3MsZXhhY3RfY291bnRzX2FuZF9maXJzdF9hcmdtYXg9VHJ1ZSxybmdfYW5kX3ZlcnNpb25zX3VuY2hhbmdlZD1UcnVlKSkKICAgICAgICAgICAgICAgIHdyaXRlKG91dCxyZXBvcnQpO2RlbCBoLGxhYmVscyxtYXNrLG5hdGl2ZSxjYW5kaWRhdGUKICAgICAgICAgICAgcmVwb3J0WydudW1lcmljYWxfY29udHJhY3QnXT0nU3RyaWN0IGNvbXBvbmVudCBnYXRlcywgbm90IHRoZSB0YXNrIHF1YWxpdHkgZ2F0ZXM7IGdyYWRpZW50IHJlbGF0aXZlIEwyIDw9MC4wMDEgYW5kIG5vcm1hbGl6ZWQgbWF4IDw9MC4wMDUgcGx1czFlLTc7IG1ldHJpY3MgcnRvbDFlLTUvYXRvbDNlLTY7IGV4YWN0IGFyZ21heC9jb3VudHMvTmFOL0luZiBjbGFzc2VzLiBGdWxsLVYgYWN0dWFsLUdyYWRTY2FsZXIgb3ZlcmZsb3cgY2xhc3NpZmljYXRpb24gaXMgY2hlY2tlZDsgdGlueSBDUFUgb3ZlcmZsb3cvdW5kZXJmbG93IHdpdG5lc3NlcyBhcmUgc2VwYXJhdGVseSBwcmVzZXJ2ZWQuIFNoYXJlZCBjb25zdW1lciBmaWxlcyBhcmUgYnl0ZS1pZGVudGljYWwgdG8gdGhlIGNvbW1vbiBtYXRlcmlhbGl6ZWQvc3RyZWFtaW5nIGNvbnN1bWVyLCBleGVyY2lzZWQgdGhyb3VnaCBub3JtYWxpemVkLWxvc3MgaWRlbnRpdHkgYWNyb3NzIGFjdHVhbCBBY2NlbGVyYXRlIGNvbnZlcnNpb247IG5vIHByb2R1Y3Rpb24vbW9kZWwgaW5zdGFsbGVyIGV4aXN0cy4nCiAgICAgICAgZWxzZToKICAgICAgICAgICAgIyBFYWNoIHNlbGVjdGVkIG1ldGhvZCBoYXMgaXRzIG93biBmcmVzaCBwcm9jZXNzLiBJbnB1dHMgYWxsb2NhdGVkIGJlZm9yZSBwZWFrcy90aW1pbmcuCiAgICAgICAgICAgIGZvciB0IGluICgxMzMsNTEyKToKICAgICAgICAgICAgICAgIGgsbGFiZWxzLG1hc2s9Zml4dHVyZSg0LHQpO2Rlbm9tPXRvcmNoLnRlbnNvcig0KnQsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5pbnQ2NCkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTtnYy5jb2xsZWN0KCk7dG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpO3RvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgICAgICAgICAgICAgYmVmb3JlPWRpY3QoYWxsb2NhdGVkPXRvcmNoLmN1ZGEubWVtb3J5X2FsbG9jYXRlZCgpLHJlc2VydmVkPXRvcmNoLmN1ZGEubWVtb3J5X3Jlc2VydmVkKCkpCiAgICAgICAgICAgICAgICBzdGFydGVkPXRpbWUucGVyZl9jb3VudGVyKCk7dmFsdWU9cnVuKGFyZ3MubWV0aG9kLGgsbGFiZWxzLG1hc2ssNjU1MzYuLGRlbm9tKTt0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7Y29sZD10aW1lLnBlcmZfY291bnRlcigpLXN0YXJ0ZWQKICAgICAgICAgICAgICAgIGFzc2VydCB0b3JjaC5pc2Zpbml0ZSh2YWx1ZVsnbG9zcyddKSBhbmQgdG9yY2guaXNmaW5pdGUodmFsdWVbJ2RoJ10pLmFsbCgpCiAgICAgICAgICAgICAgICBwZWFrPWRpY3QoYWxsb2NhdGVkPXRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSxyZXNlcnZlZD10b3JjaC5jdWRhLm1heF9tZW1vcnlfcmVzZXJ2ZWQoKSkKICAgICAgICAgICAgICAgIGRlbCB2YWx1ZTtnYy5jb2xsZWN0KCkKICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKDEpOgogICAgICAgICAgICAgICAgICAgIHZhbHVlPXJ1bihhcmdzLm1ldGhvZCxoLGxhYmVscyxtYXNrLDY1NTM2LixkZW5vbSk7ZGVsIHZhbHVlCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7c2FtcGxlcz1bXQogICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMyk6CiAgICAgICAgICAgICAgICAgICAgYmVnaW4sZW5kPXRvcmNoLmN1ZGEuRXZlbnQoZW5hYmxlX3RpbWluZz1UcnVlKSx0b3JjaC5jdWRhLkV2ZW50KGVuYWJsZV90aW1pbmc9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICBiZWdpbi5yZWNvcmQoKTt2YWx1ZT1ydW4oYXJncy5tZXRob2QsaCxsYWJlbHMsbWFzayw2NTUzNi4sZGVub20pO2VuZC5yZWNvcmQoKTtlbmQuc3luY2hyb25pemUoKQogICAgICAgICAgICAgICAgICAgIHNhbXBsZXMuYXBwZW5kKGJlZ2luLmVsYXBzZWRfdGltZShlbmQpKTtkZWwgdmFsdWUKICAgICAgICAgICAgICAgIG9ic2VydmVkX3BlYWs9ZGljdChhbGxvY2F0ZWQ9dG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpLHJlc2VydmVkPXRvcmNoLmN1ZGEubWF4X21lbW9yeV9yZXNlcnZlZCgpKQogICAgICAgICAgICAgICAgcmVwb3J0Wydyb3dzJ10uYXBwZW5kKGRpY3QoQj00LFQ9dCxWPTE1MTkzNixIPTE1MzYsdG9rZW5fY2h1bms9MTI4LHNjYWxlPTY1NTM2Lixjb2xkX2ZpcnN0X3NoYXBlX2NhbGxfd2FsbF9zPWNvbGQsd2FybV9jdWRhX2V2ZW50X21zPXNhbXBsZXMsd2FybV9tZWRpYW5fbXM9c3RhdGlzdGljcy5tZWRpYW4oc2FtcGxlcyksaW5wdXRfYmFzZWxpbmVfYnl0ZXM9YmVmb3JlLGNvbXBvbmVudF9maXJzdF9jYWxsX2Fic29sdXRlX3BlYWtfYnl0ZXM9cGVhayxjb21wb25lbnRfb2JzZXJ2ZWRfYWJzb2x1dGVfcGVha19ieXRlcz1vYnNlcnZlZF9wZWFrLGNvbXBvbmVudF9pbmNyZW1lbnRhbF9hbGxvY2F0ZWRfcGVha19ieXRlcz1vYnNlcnZlZF9wZWFrWydhbGxvY2F0ZWQnXS1iZWZvcmVbJ2FsbG9jYXRlZCddKSkKICAgICAgICAgICAgICAgIHdyaXRlKG91dCxyZXBvcnQpO2RlbCBoLGxhYmVscyxtYXNrLGRlbm9tCiAgICAgICAgICAgIHJlcG9ydFsnbWVhc3VyZW1lbnRfY29udHJhY3QnXT0nQ29sZCBzeW5jaHJvbml6ZWQgaGVhZCttZXRyaWNzK2Fkam9pbnQgd2FsbCBpbmNsdWRlcyBkaXNwYXRjaCwgYWxsb2NhdGlvbiwgQ1VEQSBsYXp5IGluaXRpYWxpemF0aW9uIGFuZCBhbnkgbGlicmFyaWVzIGZpcnN0LXVzZTsgbm8gVHJpdG9uIGNvbXBpbGUgYXR0cmlidXRpb24uIFdhcm0gc3RyZWFtIGV2ZW50cyBpbmNsdWRlIFB5dGhvbiBnYXBzIGFuZCByZXF1aXJlZCBtZXRyaWMgaG9zdCBzeW5jcy4gUGVhayBpcyB0aGlzIGlzb2xhdGVkIHN5bnRoZXRpYyBjb21wb25lbnQgSU5DTFVESU5HIGl0cyBmcm96ZW4gVy9pbnB1dCBhbGxvY2F0aW9ucywgTk9UIGZ1bGwgbW9kZWwvdHJhaW5pbmcgcGVhayBvciBleHRlcm5hbCBudmlkaWEtc21pIGFkbWlzc2lvbiBtZWFzdXJlbWVudC4gTmF0aXZlL2NodW5rIHByb2Nlc3Mgb3JkZXIgZml4ZWQ7IG5vIHRhc2stc3BlZWQgY2xhaW0uIE5hdGl2ZSBUUkwgZW50cm9weSBhbHJlYWR5IHVzZXMxMjggdG9rZW4gY2h1bmtzLiBObyBhY3R1YWwgbW9kZWwgd2VpZ2h0cy9kYXRhIGxvYWRlZC4nCiAgICAgICAgYXNzZXJ0IGZsYWdzPT1kaWN0KG1hdG11bF9hbGxvd190ZjMyPXRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzIsZnAxNl9yZWR1Y2VkX3ByZWNpc2lvbl9yZWR1Y3Rpb249dG9yY2guYmFja2VuZHMuY3VkYS5tYXRtdWwuYWxsb3dfZnAxNl9yZWR1Y2VkX3ByZWNpc2lvbl9yZWR1Y3Rpb24pCiAgICAgICAgcmVwb3J0WydzdGF0dXMnXT0nUEFTU0VEX0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJwogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOnJlcG9ydC51cGRhdGUoZXJyb3I9cmVwcihleGMpLHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpKQogICAgZmluYWxseTp3cml0ZShvdXQscmVwb3J0KQogICAgcHJpbnQoanNvbi5kdW1wcyhkaWN0KG1ldGhvZD1hcmdzLm1ldGhvZCxzdGF0dXM9cmVwb3J0WydzdGF0dXMnXSxyb3dzPWxlbihyZXBvcnRbJ3Jvd3MnXSkpKSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdCgwIGlmIHJlcG9ydFsnc3RhdHVzJ109PSdQQVNTRURfSEVBRF9DT01QT05FTlRfUFJFRkxJR0hUX09OTFknIGVsc2UgMSkKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==', 'native_source_manifest.json': 'ewogICJsb3NzX3V0aWxzLnB5IjogIjgzZGIxNmIyNGNlNWMzYTA2NDIwOTc2MjRhYTlhMGFlN2ViNzI0NDVjNDFkOGI0ZDUwNTlhMTI5ODYyZmRiYmUiLAogICJ1dGlscy5weSI6ICJlMWE2YWNkNDJjMTY5ZjczMTA1NmJiMDJiMTRmMDA5NjAwOGM2ZGVmOTc3ODIyZjMyNWMxYTdiZjFmNjQ1ZDFmIiwKICAicmV3aW5kX2hmLnB5IjogIjFmZTk1NDJlMWFiNjM0ZDFlNmYzY2U5ZGM1MGYyODg3MmQwMDk2Y2QwZjdmZGQyYmUzNDRjY2UxMGFmZDU2MDQiLAogICJvcGVyYXRpb25zLnB5IjogIjk3YTJlNThkNDkyODQ5MTI2OWM1NzM5Y2M3MjNlODRjNjQ4N2Q3OTQ2MmQ4OWQ4NmE5MmZlYjRmZjczYjdiNzMiCn0K', 'shared_consumer_manifest.json': 'ewogICJicmlkZ2Vfc3RhdGUucHkiOiAiZTNiYTY0NDQ2YThiYjNlYjhjZDZlYzMxYjRlNWQwMDE4YTQ3YzhmM2I5MGQwMGRhOTU1ZGMyZTQ0NGViMThlOSIsCiAgImNvbnN1bWVyLnB5IjogImYwMThhOWI2ODA4ZDc2MzgxYTIwNTU4MGU4YTJmZjlkMmMwZWI3MDZmYTM0Yzg0MzRmZTQxYjBhZDk5ZGEyYmMiLAogICJfX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1Igp9Cg==', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvY2h1bmsgZnJvemVuLWhlYWQgSEVBRF9DT01QT05FTlRfUFJFRkxJR0hUX09OTFkgb3JjaGVzdHJhdGlvbi4KCk5ldmVyIGluc3RhbGxzIGludG8gU09VUF9SRVBPIG9yIGNhbGxzIGEgdHJhaW5pbmcgaGFybmVzcy4gRXhpc3Rpbmcgc291cmNlL3J1bnRpbWUsCmxvY2ssIHBpZGZkL3N1YnJlYXBlciwgcXVhcmFudGluZSBhbmQgaWRsZS1jaGVjayBoZWxwZXJzIGFyZSBwcmVzZXJ2ZWQgdW5jaGFuZ2VkLgpUaGUgZGVwbG95ZWQgc2VydmljZSdzIGxpbWl0cyByZW1haW4gYXV0aG9yaXRhdGl2ZTsgZWFjaCBHUFUgY2hpbGQgZ2V0cyA8PTgwIHMuCiIiIgppbXBvcnQgZmNudGwsIGhhc2hsaWIsIGpzb24sIG9zLCBzeXMsIHRpbWUsIHV1aWQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwcm9jZXNzX2d1YXJkCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQpmcm9tIHJldmlld2VkX3N1cHBvcnQgaW1wb3J0IHZlcmlmeV9vZmZpY2lhbCwgaWRsZV9jaGVjaywgRXhwb3J0ZXIsIGV4cG9ydF9zbWFsbCwgcmVhZCwgd3JpdGUsIHNoYQoKTE9DSyA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2snKQpNQVJLRVIgPSBQYXRoKCcvdG1wL3NvdXAtZ3B1MC5sb2NrLmJsb2NrZWQnKQoKZGVmIG1haW4oKToKICAgIGhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CiAgICBpZiBub3Qgb3MuZW52aXJvbi5nZXQoJ1NPVVBfUkVQTycpIG9yIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVNVTFRTJyk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgnUmVxdWlyZXMgZGVwbG95ZWQgU09VUF9SRVBPL1NPVVBfUkVTVUxUUywgbm8gbWFudWFsIGZhbGxiYWNrJykKICAgIHJlcG8sIHJlc3VsdHMgPSBQYXRoKG9zLmVudmlyb25bJ1NPVVBfUkVQTyddKS5hYnNvbHV0ZSgpLCBQYXRoKG9zLmVudmlyb25bJ1NPVVBfUkVTVUxUUyddKS5hYnNvbHV0ZSgpCiAgICBpZiByZXBvLmlzX3N5bWxpbmsoKSBvciByZXN1bHRzLmlzX3N5bWxpbmsoKSBvciByZXBvID09IHJlc3VsdHM6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgnUHJlcGFyZWQgY2hlY2tvdXQvcmVzdWx0cyBtdXN0IGJlIHNlcGFyYXRlIHJlYWwgZGlyZWN0b3JpZXMnKQogICAgYXNzZXJ0IFBhdGgoJy9rYWdnbGUvd29ya2luZycpLmlzX2RpcigpCiAgICByZXN1bHRzLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgaWRlbnRpdHkgPSAndG9rZW4tY2h1bmstaGVhZC1jb21wb25lbnQtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J0hFQURfQ09NUE9ORU5UX1BSRUZMSUdIVF9PTkxZJywKICAgICAgICAgICAgICAgICAgdHJhaW5pbmdfZXhlY3V0ZWQ9RmFsc2UsY29tcGxldGVfZml4ZWRfdGFzaz1GYWxzZSxzY29yZT1Ob25lLGFkbWlzc2lvbj0nVU5LTk9XTicsb3JpZ2luYWxfdHJpdG9uX2NyZWRpdD1GYWxzZSwKICAgICAgICAgICAgICAgICAgbWV0aG9kX3ZhcmlhbnRzPXsnbmF0aXZlJzonbmF0aXZlX2FtcF9saW5lYXJfbmF0aXZlX2xvc3NfbWV0cmljcycsJ2NodW5rJzonb3JpZ2luYWxfdG9rZW5fY2h1bmtzX3JlY29tcHV0ZV9zY2FsZWRfYmFja3dhcmRfc2hhcmVkX2NvbnN1bWVyJ30sCiAgICAgICAgICAgICAgICAgIHBlcl9ncHVfY2hpbGRfd2FsbF9jYXBfcz04MCxjYXBfYnlwYXNzPUZhbHNlLAogICAgICAgICAgICAgICAgICBjYW5kaWRhdGVfc291cmNlX3NoYTI1Nj1yZWFkKGhlcmUvJ3NvdXJjZV9tYW5pZmVzdC5qc29uJyksCiAgICAgICAgICAgICAgICAgIG1ldGhvZHNfaW5fZXhlY3V0aW9uX29yZGVyPVsnbmF0aXZlJywnY2h1bmsnXSxwcm9jZXNzZXM9e30pCiAgICB3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICBsb2NrZmQgPSBOb25lOyBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgdHJ5OgogICAgICAgIGZvciBuYW1lLGRpZ2VzdCBpbiByZWFkKGhlcmUvJ3BheWxvYWRfbWFuaWZlc3QuanNvbicpLml0ZW1zKCk6CiAgICAgICAgICAgIHAgPSBoZXJlL25hbWUKICAgICAgICAgICAgYXNzZXJ0IG5vdCBwLmlzX3N5bWxpbmsoKSBhbmQgcC5pc19maWxlKCkgYW5kIHNoYShwKT09ZGlnZXN0LCAnUGF5bG9hZCBjaGFuZ2VkOiAnK25hbWUKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgcmVxdWlyZXMgb3BlcmF0b3IgcmVjb25jaWxpYXRpb24nKQogICAgICAgIGxvY2tmZCA9IG9zLm9wZW4oTE9DSyxvcy5PX0NSRUFUfG9zLk9fUkRXUnxvcy5PX05PRk9MTE9XLDBvNjAwKQogICAgICAgIGZjbnRsLmZsb2NrKGxvY2tmZCxmY250bC5MT0NLX0VYfGZjbnRsLkxPQ0tfTkIpCiAgICAgICAgaWYgTUFSS0VSLmV4aXN0cygpOiByYWlzZSBSdW50aW1lRXJyb3IoJ0dQVSBxdWFyYW50aW5lIGFwcGVhcmVkIHdoaWxlIGxvY2tpbmcnKQogICAgICAgIHRyZWUucHJlcGFyZSgpOyB0cmVlLmJlZ2luX3F1YXJhbnRpbmUoTUFSS0VSKQogICAgICAgIG9mZmljaWFsID0gcmVhZChoZXJlLydvZmZpY2lhbF9tYW5pZmVzdC5qc29uJykKICAgICAgICB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxvZmZpY2lhbCkKICAgICAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24sSEZfSFVCX09GRkxJTkU9JzEnLEhGX0RBVEFTRVRTX09GRkxJTkU9JzEnLFRSQU5TRk9STUVSU19PRkZMSU5FPScxJywKICAgICAgICAgICAgICAgICAgIFBZVEhPTkRPTlRXUklURUJZVEVDT0RFPScxJyxQWVRIT05VTkJVRkZFUkVEPScxJykKICAgICAgICBlbnYucG9wKCdTT1VQX1JPVEFSWV9DT0RFR0VOX0RJUicsTm9uZSkKICAgICAgICB3aXRoIEV4cG9ydGVyKHJ1bixleHBvcnQpOgogICAgICAgICAgICBwcm9iZSA9IHByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLydydW50aW1lX3Byb2JlLnB5Jyksc3RyKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJyldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1yZXBvLGVudj1lbnYsbG9nPXJ1bi8ncnVudGltZV9wcm9iZS5sb2cnLHRpbWVvdXQ9NDAsbG9ja19mZD1sb2NrZmQpCiAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ3J1bnRpbWVfcHJvYmUnXSA9IHByb2JlCiAgICAgICAgICAgIGlmIHByb2JlWydleGl0X2NvZGUnXSBvciBub3QgcmVhZChydW4vJ3J1bnRpbWVfcHJvYmUuanNvbicpLmdldCgncGFzc2VkJyk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0V4YWN0IGRlcGxveWVkIHJ1bnRpbWUgZ2F0ZSBmYWlsZWQ7IG5vIGJlbmNobWFyaycpCiAgICAgICAgICAgIGRlZiBpZGxlKGxhYmVsKToKICAgICAgICAgICAgICAgIGZvbGRlcj1ydW4vKCdpZGxlXycrbGFiZWwpO2ZvbGRlci5ta2RpcihleGlzdF9vaz1GYWxzZSkKICAgICAgICAgICAgICAgIHJlc3VsdD1pZGxlX2NoZWNrKGVudixyZXBvLGZvbGRlcixsb2NrZmQpCiAgICAgICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddWydpZGxlXycrbGFiZWxdPXJlc3VsdAogICAgICAgICAgICAgICAgaWYgcmVzdWx0WydleGl0X2NvZGUnXTpyYWlzZSBSdW50aW1lRXJyb3IoJ0lkbGUgZ2F0ZSBmYWlsZWQ6ICcrbGFiZWwpCiAgICAgICAgICAgIGlkbGUoJ2JlZm9yZV9wcmVmbGlnaHQnKQogICAgICAgICAgICAjIFNvdXJjZS1ib3VuZCBuYXRpdmUgQU1QL2Z1bGwtdm9jYWJ1bGFyeSBudW1lcmljYWwgZ2F0ZSBpbiBhIHNlcGFyYXRlIHByb2Nlc3MuCiAgICAgICAgICAgIHByZWVudiA9IGRpY3QoZW52LFRSSVRPTl9DQUNIRV9ESVI9c3RyKHJ1bi8ncHJlZmxpZ2h0X3RyaXRvbicpLAogICAgICAgICAgICAgICAgICAgICAgICAgIFRPUkNISU5EVUNUT1JfQ0FDSEVfRElSPXN0cihydW4vJ3ByZWZsaWdodF9pbmR1Y3RvcicpKQogICAgICAgICAgICBwcmUgPSBwcm9jZXNzX2d1YXJkLnJ1bihbc3lzLmV4ZWN1dGFibGUsJy1CJyxzdHIoaGVyZS8nd29ya2VyLnB5JyksJy0tbWV0aG9kJywnbnVtZXJpY2FsJywnLS1vdXQnLHN0cihydW4vJ3ByZWZsaWdodC5qc29uJyldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjd2Q9aGVyZSxlbnY9cHJlZW52LGxvZz1ydW4vJ3ByZWZsaWdodC5sb2cnLHRpbWVvdXQ9ODAsbG9ja19mZD1sb2NrZmQpCiAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ3ByZWZsaWdodCddID0gcHJlCiAgICAgICAgICAgIGlmIHByZVsnZXhpdF9jb2RlJ10gb3IgcmVhZChydW4vJ3ByZWZsaWdodC5qc29uJykuZ2V0KCdzdGF0dXMnKSE9J1BBU1NFRF9IRUFEX0NPTVBPTkVOVF9QUkVGTElHSFRfT05MWSc6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBBTVAgZnVsbC12b2NhYnVsYXJ5IGhlYWQgbnVtZXJpY2FsIGdhdGUgZmFpbGVkOyBubyBiZW5jaG1hcmsnKQogICAgICAgICAgICBzdGF0dXNbJ3N0YXR1cyddPSdSVU5OSU5HX01JQ1JPQkVOQ0hfT05MWSc7d3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgICAgICBpZGxlKCdwcmVmbGlnaHQnKQogICAgICAgICAgICBmb3IgbWV0aG9kIGluIHN0YXR1c1snbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXInXToKICAgICAgICAgICAgICAgIGNhY2hlPXJ1bi8ndHJpdG9uX2NhY2hlJy9tZXRob2Q7Y2FjaGUubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgY2hpbGRlbnY9ZGljdChlbnYsVFJJVE9OX0NBQ0hFX0RJUj1zdHIoY2FjaGUpLFRPUkNISU5EVUNUT1JfQ0FDSEVfRElSPXN0cihydW4vJ2luZHVjdG9yX2NhY2hlJy9tZXRob2QpKQogICAgICAgICAgICAgICAgcHJvY2Vzcz1wcm9jZXNzX2d1YXJkLnJ1bihbc3lzLmV4ZWN1dGFibGUsJy1CJyxzdHIoaGVyZS8nd29ya2VyLnB5JyksJy0tbWV0aG9kJyxtZXRob2QsJy0tb3V0JyxzdHIocnVuLyhtZXRob2QrJy5qc29uJykpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPWhlcmUsZW52PWNoaWxkZW52LGxvZz1ydW4vKG1ldGhvZCsnLmxvZycpLHRpbWVvdXQ9ODAsbG9ja19mZD1sb2NrZmQpCiAgICAgICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddW21ldGhvZF09cHJvY2VzcwogICAgICAgICAgICAgICAgaWYgcHJvY2Vzc1snZXhpdF9jb2RlJ10gb3IgcmVhZChydW4vKG1ldGhvZCsnLmpzb24nKSkuZ2V0KCdzdGF0dXMnKSE9J1BBU1NFRF9IRUFEX0NPTVBPTkVOVF9QUkVGTElHSFRfT05MWSc6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdJbmNvbXBsZXRlL2ZhaWxlZC9jZW5zb3JlZCBtaWNyb2JlbmNobWFyazogJyttZXRob2QpCiAgICAgICAgICAgICAgICBpZGxlKG1ldGhvZCkKICAgICAgICAgICAgICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICAgICAgdmVyaWZ5X29mZmljaWFsKHJlcG8sb2ZmaWNpYWwpCiAgICAgICAgICAgIHJlcG9ydHM9e21ldGhvZDpyZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKSBmb3IgbWV0aG9kIGluIHN0YXR1c1snbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXInXX0KICAgICAgICAgICAgYXNzZXJ0IGFsbChsZW4oclsncm93cyddKT09MiBhbmQgclsndHJhaW5pbmdfZXhlY3V0ZWQnXSBpcyBGYWxzZSBhbmQgclsnY29tcGxldGVfZml4ZWRfdGFzayddIGlzIEZhbHNlIGZvciByIGluIHJlcG9ydHMudmFsdWVzKCkpCiAgICAgICAgICAgIHdyaXRlKHJ1bi8nY29tcGFyaXNvbi5qc29uJyxkaWN0KHNjb3BlPSdIRUFEX0NPTVBPTkVOVF9QUkVGTElHSFRfT05MWScscmVwb3J0cz1yZXBvcnRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNjb3JlPU5vbmUsYWRtaXNzaW9uPSdVTktOT1dOJyxjb21wbGV0ZV9maXhlZF90YXNrPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5vdGU9J0lzb2xhdGVkIHN5bnRoZXRpYyBoZWFkK25hdGl2ZS1yZXF1aXJlZC1tZXRyaWNzK2Fkam9pbnQgb25seS4gTmF0aXZlL2NodW5rIGFyZSBmcmVzaCBwcm9jZXNzZXMgd2l0aCBzYW1lIHN5bnRoZXRpYyBmcm96ZW4gRlAzMiBQYXJhbWV0ZXIuIEFic29sdXRlIGFsbG9jYXRvciBwZWFrcyBpbmNsdWRlIGNvbXBvbmVudCBXL2lucHV0cywgbm90IGEgbW9kZWwgb3IgZnVsbCB0cmFpbmluZyBwZWFrLiBObyB0YXNrIHNwZWVkdXAsIDNHQi1tb2RlbCBjbGFpbSwgb3JpZ2luYWwgVHJpdG9uIGNyZWRpdCwgb3Igb3duZXJzaGlwLXNoYXJlIGluZmVyZW5jZS4nKSkKICAgICAgICAgICAgc3RhdHVzWydzdGF0dXMnXT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzdGF0dXMudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9GQUlMRURfT1JfQ0VOU09SRURfTUlDUk9CRU5DSF9PTkxZJyxlcnJvcj1yZXByKGV4YykpCiAgICAgICAgaWYgbG9ja2ZkIGlzIG5vdCBOb25lOnRyZWUucXVhcmFudGluZSgnVG9rZW4tY2h1bmsgaGVhZCBjb21wb25lbnQgZmFpbGVkOiAnK3JlcHIoZXhjKSkKICAgIGZpbmFsbHk6CiAgICAgICAgc3RhdHVzWydvcmNoZXN0cmF0b3Jfd2FsbF9zJ109dGltZS5tb25vdG9uaWMoKS1zdGFydGVkCiAgICAgICAgZXhwb3J0X3NtYWxsKHJ1bixleHBvcnQpO3dyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICBpZiBsb2NrZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyZWUuZmluaXNoX3F1YXJhbnRpbmUoKTtvcy5jbG9zZShsb2NrZmQpCiAgICBwcmludChqc29uLmR1bXBzKHsnc2NvcGUnOidIRUFEX0NPTVBPTkVOVF9QUkVGTElHSFRfT05MWScsJ3N0YXR1cyc6c3RhdHVzWydzdGF0dXMnXSwnZXhwb3J0JzpzdHIoZXhwb3J0KSwKICAgICAgICAgICAgICAgICAgICAgICdzY29yZSc6Tm9uZSwnYWRtaXNzaW9uJzonVU5LTk9XTid9KSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdCgwIGlmIHN0YXR1c1snc3RhdHVzJ109PSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJyBlbHNlIDEpCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI5NDc5NGQ0ZjRhYjRiYTc2ZWI1OGY3MzM4MzAwOGVhNjA1ZTNhNmUyNWFmYjkxZjU1MDcyM2I1YzgwMWUwM2FhIiwKICAic29sdXRpb24vdG9rZW5fY2h1bmtfaGVhZC5weSI6ICJjNzVkMzNiN2FkZmE0MmU2NTljMDQ5Y2U5OTMzYTU3NTAxMzI0YmFiN2Q3YWY4YjNiYmRhMDgzMmJjZTczZmI1IiwKICAic2hhcmVkX2NvbnN1bWVyL19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLAogICJzaGFyZWRfY29uc3VtZXIvYnJpZGdlX3N0YXRlLnB5IjogImUzYmE2NDQ0NmE4YmIzZWI4Y2Q2ZWMzMWI0ZTVkMDAxOGE0N2M4ZjNiOTBkMDBkYTk1NWRjMmU0NDRlYjE4ZTkiLAogICJzaGFyZWRfY29uc3VtZXIvY29uc3VtZXIucHkiOiAiZjAxOGE5YjY4MDhkNzYzODFhMjA1NTgwZThhMmZmOWQyYzBlYjcwNmZhMzRjODQzNGZlNDFiMGFkOTlkYTJiYyIKfQo=', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI5NDc5NGQ0ZjRhYjRiYTc2ZWI1OGY3MzM4MzAwOGVhNjA1ZTNhNmUyNWFmYjkxZjU1MDcyM2I1YzgwMWUwM2FhIiwKICAic29sdXRpb24vdG9rZW5fY2h1bmtfaGVhZC5weSI6ICJjNzVkMzNiN2FkZmE0MmU2NTljMDQ5Y2U5OTMzYTU3NTAxMzI0YmFiN2Q3YWY4YjNiYmRhMDgzMmJjZTczZmI1IiwKICAic2hhcmVkX2NvbnN1bWVyL19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLAogICJzaGFyZWRfY29uc3VtZXIvYnJpZGdlX3N0YXRlLnB5IjogImUzYmE2NDQ0NmE4YmIzZWI4Y2Q2ZWMzMWI0ZTVkMDAxOGE0N2M4ZjNiOTBkMDBkYTk1NWRjMmU0NDRlYjE4ZTkiLAogICJzaGFyZWRfY29uc3VtZXIvY29uc3VtZXIucHkiOiAiZjAxOGE5YjY4MDhkNzYzODFhMjA1NTgwZThhMmZmOWQyYzBlYjcwNmZhMzRjODQzNGZlNDFiMGFkOTlkYTJiYyIsCiAgInByb2Nlc3NfZ3VhcmQucHkiOiAiODI0NjU5NDNhM2E5YWYzODMxYTUzOTlhM2E1NWQwOWFjMjkwYmE3MzY1YzRlODBkNmJjNWU3OWZhYjYyYTZmZiIsCiAgIm93bmVkX3Byb2Nlc3NfdHJlZS5weSI6ICI2OTNkZGU3MDhkYTVjMDYwNTNhZjIzOTQ2Mzk1MjExNWVmNWU4ODY1ZjUwYmNiYzUxYTIwMzgyMzdjYTU3ZDY5IiwKICAicnVudGltZV9wcm9iZS5weSI6ICJlZmUxYjdjZjNmM2NmMWUzN2JiYzE2OWE4NDY4ODU5YzUxMjdlYzMyODFhMzdlMzkwNjg3MDNiMmViM2MxNDgwIiwKICAib2ZmaWNpYWxfbWFuaWZlc3QuanNvbiI6ICJkMDEzZTE2MzM5NDBjMjdkNzA4ZTI0NzRhODI4MDk5MDAwMGUzMzAxOGU4ODllZDE0NjU1Yzg1NjMyOWZmMWQ4IiwKICAicmV2aWV3ZWRfc3VwcG9ydC5weSI6ICI3YzVkNjAyZDA3MTM4M2VmMDdmZGRkOTk2NzBlZDFjY2UzMTQxZmVhYmQ1YWY4Mzk0ZjA1M2M2YjRlMDcxMjJjIiwKICAid29ya2VyLnB5IjogIjQzNTc1OWEzNDEwOGFjMGQ3M2JhNTFmYmM2Y2Y5MjUxYTdiMThhYjk4Mjk2YWU2OTU1MzJhZjhjZTNkNDlkN2QiLAogICJuYXRpdmVfc291cmNlX21hbmlmZXN0Lmpzb24iOiAiMTJkZTY5OTFmZDQxZjJkZWE2Y2E4OTMwMDc5ZjEwYmEyY2Q3Yzc1MmY5NTU4MzZlMzc3NGYzOTk3NjY2YTgxNiIsCiAgInNoYXJlZF9jb25zdW1lcl9tYW5pZmVzdC5qc29uIjogIjdjMzI4MzNiZmVjN2Y4ZDE3ZWM2NzcxZTk2ODNiOTIyMTA5ZjJlMWMwOWU3ZTQ3MzU5YTg2YzAzYWQwZmYzOTUiLAogICJkcml2ZXIucHkiOiAiYWJmZDhkOWEyMjY3NjdlZDI3NGQyZTRhNGUwZWQ3MzU5ZTI5ZDVmYmNiY2ZlNDM3MWI4ZDljMzRmYzcxMDcyYSIsCiAgInNvdXJjZV9tYW5pZmVzdC5qc29uIjogIjdhMzA1YzVjYTliYTBiMzdlY2I1Nzg5YmY5M2U4YzEyNjQwMzA3ZGQ1ZjZjNzNlMTEwZDI0MzllZWZjYmFjOGEiCn0K'}
PAYLOAD_SHA256 = {'solution/__init__.py': '94794d4f4ab4ba76eb58f73383008ea605e3a6e25afb91f550723b5c801e03aa', 'solution/token_chunk_head.py': 'c75d33b7adfa42e659c049ce9933a57501324bab7d7af8b3bbda0832bce73fb5', 'shared_consumer/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'shared_consumer/bridge_state.py': 'e3ba64446a8bb3eb8cd6ec31b4e5d0018a47c8f3b90d00da955dc2e444eb18e9', 'shared_consumer/consumer.py': 'f018a9b6808d76381a205580e8a2ff9d2c0eb706fa34c8434fe41b0ad99da2bc', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': '435759a34108ac0d73ba51fbc6cf9251a7b18ab98296ae695532af8ce3d49d7d', 'native_source_manifest.json': '12de6991fd41f2dea6ca8930079f10ba2cd7c752f955836e3774f3997666a816', 'shared_consumer_manifest.json': '7c32833bfec7f8d17ec6771e9683b922109f2e1c09e7e47359a86c03ad0ff395', 'driver.py': 'abfd8d9a226767ed274d2e4a4e0ed7359e29d5fbcbcfe4371b8d9c34fc71072a', 'source_manifest.json': '7a305c5ca9ba0b37ecb5789bf93e8c12640307dd5f6c73e110d2439eefcbac8a', 'payload_manifest.json': '79600b5ccfabf12c0bdc7c9a969f9bff93a8b7dd033c293ebd34f10bd4bb114a'}
STAGED=Path(tempfile.mkdtemp(prefix='token-chunk-head-component-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
